# GAAPO with local Ollama — corrected implementation

Re-implementation and audit of **Sécheresse, Guilbert‑Ly & Villedieu de Torcy, "GAAPO: genetic
algorithm applied to prompt optimization", *Front. Artif. Intell.* 8:1613007 (2025)**, applied to
multilabel ETHOS hate-speech classification.

The notebook evolves a classification prompt with the paper's five generation strategies
(random mutation 40 %, APO 20 %, OPRO 20 %, few-shot 10 %, crossover 10 %), evaluates candidates
on a validation split, and reports a held-out test score with the paper's **strict set-equality
accuracy**.

This file is a corrected copy of `GAAPO.ipynb` (which is left untouched). The original had a number
of defects that changed *what was measured* — duplicate candidates collapsing into one evaluation,
a baseline scored on 10 examples but compared against candidates scored on 50, an unimplemented
successive-halving rule, no RNG control anywhere, and an ablation whose differences were pure RNG
noise. Section 15 documents every change. Section 16 explains how to reproduce the experiments.

> **Verification status.** No full-scale run is embedded in this file. Every number must be
> regenerated by running the cells. The artifacts in `content/gaapo_outputs/` were produced by the
> *original* notebook with population 10 and 6 generations; they are **not** reproducible from this
> file and are kept only as historical evidence.

## Connect to Ollama

**Default (recommended): run this notebook on the same machine as Ollama.** `OLLAMA_BASE_URL`
defaults to `http://localhost:11434`, which is what the code below assumes.

If you must run the kernel elsewhere (e.g. Colab) and reach Ollama through a tunnel:

1. On the Ollama host: `ollama serve` (or the desktop app) plus a tunnel, e.g.
   `ngrok http 11434`.
2. In this notebook set `OLLAMA_BASE_URL` to the tunnel URL **and** supply credentials. For an
   ngrok *basic-auth* policy add `OLLAMA_AUTH_HEADERS = {"Authorization": "Basic <base64>"}` or set
   `OLLAMA_API_KEY` to the token your gateway expects. The original notebook described this
   Colab/ngrok flow but had no authentication code at all, so the remote path could not work.
3. Treat the URL/token as a secret: put them in environment variables
   (`OLLAMA_BASE_URL`, `OLLAMA_API_KEY`, `GAAPO_*`) rather than committing them.

Nothing is downloaded by this notebook; model pulls are left to you.

## 1. Install dependencies

Only needed once per kernel. `datasets` pulls in `huggingface_hub` and `pyarrow`, which the ETHOS
loader below relies on explicitly. `tabulate` is used for markdown report tables.

In [ ]:
%pip install -q "openai>=1.40" "httpx>=0.27" "pandas>=2.0" "datasets>=2.19" "matplotlib>=3.8" "tabulate>=0.9" "huggingface_hub>=0.23" "pyarrow>=15" "requests>=2.31"

import importlib.metadata as _md
for _pkg in ("openai", "httpx", "pandas", "datasets", "matplotlib", "tabulate",
             "huggingface_hub", "pyarrow", "requests"):
    try:
        print(f"{_pkg:18s} {_md.version(_pkg)}")
    except Exception:
        print(f"{_pkg:18s} MISSING")

## 2. Configuration

Every experimental knob lives here. The defaults differ from the original notebook in three
deliberate, documented ways:

| Setting | Original | Now | Why |
|---|---|---|---|
| `MODEL` (single role) | `phi3.5:latest` for both roles | `qwen2.5:7b` optimizer + `qwen2.5:3b` target | The paper separates *optimizer* and *target* LLMs (§2.3.4.1); one model for both conflates them. The old optimizer echoed the seed prompt back (see `best_prompt.json`). |
| `APO_ERROR_SOURCE` | validation errors | `"train"` | Paper §2.3.3.1: the **train** set is used for "strategy-specific optimization … APO leverages this set for error analysis". Using validation errors for generation and then validating on the same set is optimistic. |
| `STRATEGY_ALLOCATION` | independent weighted draws per slot | `"quota"` | Paper §2.3.3.2: "#candidates per strategy = weight × population size; remaining slots to the highest weight". |

`RUN_*` switches let you skip expensive stages. `ABLATION_N_REPEATS = 3` follows the paper's
3-repetition protocol for direct comparisons (§2.3.4.1).

**Compute reality check (measured on this machine).** With `ALL` evaluation a run costs
`50 + generations × population × |val|` target calls plus one optimizer call per candidate. The
**optimizer is the bottleneck**: `qwen2.5:7b` needs roughly 20–30 s to emit one prompt on this M1
(CPU), while a `qwen2.5:3b` classification call is ~1–2 s and four of them run concurrently. A
measured smoke run — 10 candidates × 2 generations on 4 validation examples — took about 4 minutes.

Scaled to the defaults below, that is roughly **2 h for the main run** and **13 h for the ablation**
(6 configurations × 3 repeats), so plan for an overnight job. To get a first answer sooner, set
`ABLATION_N_REPEATS = 1`, `ABLATION_GENERATIONS = 3`, and/or use a smaller optimizer model; the
smoke test in section 10 takes a few minutes.

In [ ]:
import os
from pathlib import Path

# ------------------------------------------------------------------ model roles
OPTIMIZER_MODEL = os.environ.get("GAAPO_OPTIMIZER_MODEL", "qwen2.5:7b")
TARGET_MODEL    = os.environ.get("GAAPO_TARGET_MODEL",    "qwen2.5:3b")

OLLAMA_BASE_URL    = os.environ.get("OLLAMA_BASE_URL", "http://localhost:11434")
OLLAMA_API_KEY     = os.environ.get("OLLAMA_API_KEY", "ollama")
# Extra headers for a proxied/authenticated tunnel, e.g.
#   {"Authorization": "Basic ..."} or {"ngrok-skip-browser-warning": "true"}
OLLAMA_EXTRA_HEADERS = {}

# ------------------------------------------------------------------ determinism
RANDOM_SEED = 42                 # data split, strategy streams, ablation repeats

# ------------------------------------------------------------------ output
_p = Path(os.environ.get("GAAPO_OUTPUT_DIR", "content/gaapo_outputs_v2"))


def _notebook_dir():
    # In Jupyter the kernel records its start directory; fall back to the CWD.
    try:
        from IPython import get_ipython
        ip = get_ipython()
        if ip is not None and getattr(ip, "starting_dir", None):
            return Path(ip.starting_dir)
    except Exception:
        pass
    return Path.cwd()


_NB_DIR = _notebook_dir()
OUTPUT_DIR = _p if _p.is_absolute() else (_NB_DIR / _p)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------ runtime
MAX_CONCURRENT_REQUESTS = 4
REQUEST_TIMEOUT_S       = 180.0
TARGET_MAX_TOKENS       = 160      # original: 150
OPTIMIZER_MAX_TOKENS    = 400      # original: 600; prompts need ~150-250 tokens
TARGET_TEMPERATURE      = 0.0

# ------------------------------------------------------------------ paper fidelity
APO_ERROR_SOURCE        = "train"  # "train" (paper) or "val"
STRATEGY_ALLOCATION     = "quota"  # "quota" (paper) or "random"
CROSSOVER_CLEANUP       = True     # LLM cleanup after the paper's word-midpoint splice
LENIENT_PARSING         = True     # keyword fallback when the model emits prose instead of JSON
MAX_GENERATION_ATTEMPTS = 3        # retries when a generated candidate fails validation

# ------------------------------------------------------------------ run switches
RUN_SMOKE_TEST            = True
RUN_MAIN_EXPERIMENT       = True
RUN_ABLATION              = True
RUN_SELECTION_COMPARISON  = False   # paper §3.7; +3 full runs
RUN_ADAPTIVE_STUDY        = True    # adaptive operator selection (§17-20)
INCLUDE_SINGLE_STRATEGY   = False   # paper Table 2/4 baselines; +5 full runs per repeat

# ------------------------------------------------------------------ scale
MAIN_POPULATION_SIZE = 20
MAIN_GENERATIONS     = 8
MAIN_EVAL_MODE       = "all"        # "all" | "successive_halving" | "bandit"

ABLATION_POPULATION_SIZE = 10
ABLATION_GENERATIONS     = 6
ABLATION_EVAL_MODE       = "all"
ABLATION_N_REPEATS       = 3
# Leave-one-out arms actually run. All six roughly double the sweep; these four
# answer the informative questions (no error feedback, no trajectory feedback,
# no non-LLM diversity source) and keep the whole study inside one night.
ABLATION_CONFIGS_TO_RUN  = ["full_hybrid", "without_apo", "without_opro",
                            "without_random_mutation"]

ADAPTIVE_POPULATION_SIZE = 10
ADAPTIVE_GENERATIONS     = 6
ADAPTIVE_EVAL_MODE       = "all"
ADAPTIVE_N_REPEATS       = 3
RESUME_ADAPTIVE          = True    # skip arm-runs already written to adaptive_runs.csv

# ---- adaptive operator selection (the paper's weights are the prior)
AOS_ALPHA = 0.30    # EWMA learning rate for the credit signal
AOS_TAU   = 0.50    # softmax temperature over Q
AOS_GAMMA = 0.10    # exploration floor on the strategy distribution

print(f"notebook dir : {_NB_DIR}")
print(f"output dir   : {OUTPUT_DIR.resolve()}")
print(f"optimizer    : {OPTIMIZER_MODEL}")
print(f"target       : {TARGET_MODEL}")

In [ ]:
import requests

def _norm_model_name(name):
    return name if ":" in name else f"{name}:latest"


def check_ollama_models():
    try:
        r = requests.get(f"{OLLAMA_BASE_URL}/api/tags",
                         headers=OLLAMA_EXTRA_HEADERS, timeout=10)
        r.raise_for_status()
        installed = [m["name"] for m in r.json().get("models", [])]
    except Exception as exc:
        raise RuntimeError(
            f"Cannot reach Ollama at {OLLAMA_BASE_URL}. Is it running? "
            f"(original error: {exc})"
        ) from exc

    wanted = {"optimizer": OPTIMIZER_MODEL, "target": TARGET_MODEL}
    missing = [n for n in wanted.values() if _norm_model_name(n) not in installed]
    if missing:
        pulls = "; ".join(f"ollama pull {m}" for m in missing)
        raise RuntimeError(
            f"Missing model(s): {missing}. Run: {pulls}. Installed: {installed}"
        )
    print("Ollama reachable.")
    for role, name in wanted.items():
        print(f"  {role:9s} -> {name}")
    print(f"  installed -> {installed}")
    return True


check_ollama_models()

## 3. Load and split ETHOS

The paper uses 300 samples of the ETHOS **multilabel** subset: 50 train (APO error analysis and
few-shot examples), 50 validation (selection), 200 test (final only). Category scores `>= 0.5`
become labels, as in the original notebook.

Two implementation notes:

* `load_dataset("ethos", "multilabel")` fails on `datasets>=4` because the Hub repo still ships an
  old-style loader script. We load the auto-converted parquet file directly, which is the same data.
* The HuggingFace conversion contains **433** rows (the paper's "443" is the non-multilabel ETHOS
  count), so the 50/50/200 split leaves 133 rows unused. The split is *shuffled*, not stratified,
  and is reproducible only for a given `pandas` major version — the manifest records its hashes.

In [ ]:
import hashlib
import pandas as pd
from datasets import load_dataset

LABELS = [
    "violence", "directed_vs_generalized", "gender", "race", "national_origin",
    "disability", "religion", "sexual_orientation",
]
LABEL_LOOKUP = {lbl.lower(): lbl for lbl in LABELS}

PARQUET_URI = ("hf://datasets/iamollas/ethos@refs/convert/parquet/"
               "multilabel/train/*.parquet")

try:
    _raw = load_dataset("parquet", data_files=PARQUET_URI)["train"]
    df_full = _raw.to_pandas()
except Exception as exc:
    print("Primary parquet load failed; trying huggingface_hub fallback:", exc)
    from huggingface_hub import hf_hub_download
    _path = hf_hub_download(
        repo_id="iamollas/ethos", repo_type="dataset",
        filename="multilabel/train/0000.parquet", revision="refs/convert/parquet",
    )
    df_full = pd.read_parquet(_path)

missing = [lbl for lbl in LABELS if lbl not in df_full.columns]
if missing or "text" not in df_full.columns:
    raise RuntimeError(
        f"Unexpected ETHOS schema. Missing labels: {missing}; "
        f"columns: {list(df_full.columns)}"
    )

# Paper: category score >= 0.5 -> label present.
df_full["labels"] = df_full.apply(
    lambda row: [lbl for lbl in LABELS if float(row[lbl]) >= 0.5], axis=1
)
df_full = (df_full.rename(columns={"text": "input"})[["input", "labels"]]
                  .sample(frac=1, random_state=RANDOM_SEED)
                  .reset_index(drop=True))

n = len(df_full)
if n < 100:
    raise RuntimeError(f"ETHOS multilabel split too small: {n} rows")

train_df, val_df = df_full.iloc[:50], df_full.iloc[50:100]
test_df = df_full.iloc[100:100 + min(200, n - 100)]

# Hard guarantee that the three splits share no row.
assert not (set(train_df.index) & set(val_df.index))
assert not (set(train_df.index) & set(test_df.index))
assert not (set(val_df.index) & set(test_df.index))

train_df, val_df, test_df = (d.reset_index(drop=True) for d in (train_df, val_df, test_df))

SPLIT_MANIFEST = {
    "dataset": "iamollas/ethos (multilabel, refs/convert/parquet)",
    "n_available": n,
    "n_train": len(train_df), "n_val": len(val_df), "n_test": len(test_df),
    "shuffle_seed": RANDOM_SEED,
    "sha256_train": hashlib.sha256("\n".join(train_df["input"]).encode()).hexdigest(),
    "sha256_val":   hashlib.sha256("\n".join(val_df["input"]).encode()).hexdigest(),
    "sha256_test":  hashlib.sha256("\n".join(test_df["input"]).encode()).hexdigest(),
}

for _name, _d in (("train", train_df), ("val", val_df), ("test", test_df)):
    _k = _d["labels"].apply(len)
    print(f"{_name:5s} n={len(_d):3d}  empty={int((_k == 0).sum()):3d}  "
          f"single={int((_k == 1).sum()):3d}  multi={int((_k >= 2).sum()):3d}  "
          f"mean_labels={_k.mean():.2f}")
print("example:", train_df.iloc[0].to_dict())

## 4. Seed prompt, parsing and metrics

`BASELINE_PROMPT` is byte-identical to the original notebook so that the new runs stay comparable
with the old artifacts.

The paper's ETHOS metric is **strict set-equality accuracy** (order-insensitive): a prediction is
correct only if the predicted label set equals the gold set. `exact_match` implements that and is
now the single metric used everywhere (the original defined it and then re-implemented it inline).

Parsing is necessarily an implementation detail, but it can silently change results, so:

* labels are matched case-insensitively and whitespace-normalised;
* a JSON **array** or an object containing a label list is accepted;
* the original keyword scan is kept but every prediction records *how* it was parsed, and the
  report prints the breakdown. If the model emits prose, `keyword`/`empty` counts expose it
  instead of hiding it. Turn it off with `LENIENT_PARSING = False`.
* `macro_f1` is reported as a secondary diagnostic only; it is **not** used for selection, so the
  optimisation objective remains the paper's strict accuracy.

In [ ]:
LABELS_LIST_STR = ", ".join(LABELS)

BASELINE_PROMPT = (
    "You are a content moderation classifier. Given a message, identify which of the "
    "following categories apply (zero or more may apply):\n"
    f"{LABELS_LIST_STR}\n\n"
    "Message: {input}\n\n"
    "Respond with ONLY a JSON array of the applicable category names from the list "
    "above, e.g. [\"violence\",\"religion\"]. If none apply, respond with []."
)

assert "{input}" in BASELINE_PROMPT
assert all(lbl in BASELINE_PROMPT for lbl in LABELS)
print(BASELINE_PROMPT)

In [ ]:
import json
import re
from collections import Counter

PARSE_COUNTS = Counter()          # how predictions were obtained, across the whole session


def strip_code_fences(text):
    t = (text or "").strip()
    t = re.sub(r"^\s*```[A-Za-z0-9_+-]*\s*", "", t)
    t = re.sub(r"\s*```\s*$", "", t)
    return t.strip()


def _labels_from_json_object(obj, lookup):
    # Returns a list of canonical labels if obj is a label collection, else None.
    if isinstance(obj, str):
        key = obj.strip().lower()
        return [lookup[key]] if key in lookup else None
    if isinstance(obj, list):
        out = []
        for item in obj:
            if isinstance(item, str):
                key = item.strip().lower()
                if key in lookup:
                    out.append(lookup[key])
        return out
    if isinstance(obj, dict):
        # Prefer an explicit label/category list, then any list of known labels.
        preferred = ("categories", "labels", "applicable_categories", "output")
        for k in preferred:
            if k in obj:
                got = _labels_from_json_object(obj[k], lookup)
                if got is not None:
                    return got
        for v in obj.values():
            got = _labels_from_json_object(v, lookup)
            if got is not None:
                return got
    return None


def parse_prediction(raw_text, valid_labels=None, lenient=None):
    # Return (set_of_labels, method). method in {json, json_substring, keyword, empty}.
    valid_labels = list(valid_labels or LABELS)
    lenient = LENIENT_PARSING if lenient is None else lenient
    lookup = {lbl.lower(): lbl for lbl in valid_labels}
    text = strip_code_fences(raw_text)
    if not text:
        PARSE_COUNTS["empty"] += 1
        return set(), "empty"

    try:
        got = _labels_from_json_object(json.loads(text), lookup)
        if got is not None:
            PARSE_COUNTS["json"] += 1
            return set(got), "json"
    except Exception:
        pass

    m = re.search(r"\[[\s\S]*?\]", text)
    if m:
        try:
            got = _labels_from_json_object(json.loads(m.group(0)), lookup)
            if got is not None:
                PARSE_COUNTS["json_substring"] += 1
                return set(got), "json_substring"
        except Exception:
            pass

    m = re.search(r"\{[\s\S]*\}", text)
    if m:
        try:
            got = _labels_from_json_object(json.loads(m.group(0)), lookup)
            if got is not None:
                PARSE_COUNTS["json_substring"] += 1
                return set(got), "json_substring"
        except Exception:
            pass

    if lenient:
        found, low = set(), text.lower()
        for lbl in valid_labels:
            if re.search(r"\b" + re.escape(lbl.lower()) + r"\b", low):
                found.add(lbl)
        if found:
            PARSE_COUNTS["keyword"] += 1
            return found, "keyword"

    PARSE_COUNTS["empty"] += 1
    return set(), "empty"


def safe_parse_label_set(raw_text, valid_labels=None):
    # Backwards-compatible helper: returns only the label set.
    return parse_prediction(raw_text, valid_labels)[0]


def exact_match(pred_set, gold_set):
    # Paper §2.3.3.4: order-insensitive set equality.
    return 1.0 if set(pred_set) == set(gold_set) else 0.0


def macro_f1(pred_sets, gold_sets, labels=None):
    labels = list(labels or LABELS)
    scores = []
    for lbl in labels:
        tp = sum(1 for p, g in zip(pred_sets, gold_sets) if lbl in p and lbl in g)
        fp = sum(1 for p, g in zip(pred_sets, gold_sets) if lbl in p and lbl not in g)
        fn = sum(1 for p, g in zip(pred_sets, gold_sets) if lbl not in p and lbl in g)
        prec = tp / (tp + fp) if tp + fp else 0.0
        rec = tp / (tp + fn) if tp + fn else 0.0
        scores.append(2 * prec * rec / (prec + rec) if prec + rec else 0.0)
    return sum(scores) / len(scores) if scores else 0.0


# quick self-check
assert parse_prediction('["Violence", " gender "]')[0] == {"violence", "gender"}
assert parse_prediction('```json\n{"categories": ["race"]}\n```')[0] == {"race"}
assert exact_match({"race", "gender"}, {"gender", "race"}) == 1.0
assert exact_match({"race"}, {"race", "gender"}) == 0.0
print("parser self-checks passed")

## 5. LLM client

One client per role. Both speak the Ollama OpenAI-compatible API and add:

* **Deterministic request cache** keyed on `(model, system, max_tokens, temperature, prompt)`, with
  an in-flight lock so two threads never issue the same request twice (the original could, and its
  duplicate candidates made that common).
* **Retries** with exponential backoff for transient errors only (429/5xx/timeouts); authentication
  and "model not found" fail immediately.
* **Reasoning-model handling.** A thinking model can return `content=""` and put everything in a
  `reasoning` channel. The client detects this, switches `reasoning_effort="none"` once, and
  retries; empty outputs and truncated responses (`finish_reason="length"`) are counted so a silent
  0 % score is visible instead of mysterious. Verified against `qwen3.5:4b` on this machine.
* **Token accounting** per role, used in the run manifest.

Note: `temperature=0.0` for the target does **not** make a local LLM bit-deterministic (batch
scheduling, quantisation and sampler details still vary). The cache makes repeated identical
requests consistent *within* a run.

In [ ]:
import hashlib
import threading
import time
from dataclasses import dataclass, field

import httpx
from openai import OpenAI


@dataclass
class LLMStats:
    calls: int = 0
    cache_hits: int = 0
    retries: int = 0
    empty_outputs: int = 0
    truncated: int = 0
    reasoning_fallbacks: int = 0
    prompt_tokens: int = 0
    completion_tokens: int = 0

    def as_dict(self):
        return dict(self.__dict__)


class LLMClient:
    # Ollama client with caching, retries and reasoning-model fallback.

    def __init__(self, model, role, base_url=None, api_key=None, extra_headers=None,
                 timeout=None, default_max_tokens=256, default_temperature=0.7,
                 disable_reasoning="auto"):
        self.model = model
        self.role = role
        self.base_url = (base_url or OLLAMA_BASE_URL).rstrip("/")
        self.default_max_tokens = default_max_tokens
        self.default_temperature = default_temperature
        self.disable_reasoning = disable_reasoning
        self.stats = LLMStats()

        self._extra_body = {}
        if disable_reasoning is True:
            self._extra_body = {"reasoning_effort": "none"}

        self._cache = {}
        self._inflight = {}
        self._lock = threading.Lock()
        self._jitter = __import__("random").Random(0)
        timeout = float(timeout or REQUEST_TIMEOUT_S)

        self._http = httpx.Client(timeout=httpx.Timeout(timeout))
        self._client = OpenAI(
            api_key=api_key or OLLAMA_API_KEY,
            base_url=f"{self.base_url}/v1",
            http_client=self._http,
            max_retries=0,
            timeout=timeout,
            default_headers=dict(extra_headers) if extra_headers else None,
        )

    # ------------------------------------------------------------------ helpers
    def _key(self, prompt, system, max_tokens, temperature):
        blob = f"{self.model}|{system or ''}|{max_tokens}|{temperature}|{prompt}"
        return hashlib.sha256(blob.encode()).hexdigest()

    def _call_once(self, prompt, system, max_tokens, temperature):
        messages = ([{"role": "system", "content": system}] if system else []) + \
                   [{"role": "user", "content": prompt}]
        kwargs = dict(model=self.model, messages=messages,
                      max_tokens=max_tokens, temperature=temperature)
        if self._extra_body:
            kwargs["extra_body"] = dict(self._extra_body)

        self.stats.calls += 1
        resp = self._client.chat.completions.create(**kwargs)
        choice = resp.choices[0]
        message = choice.message
        content = (message.content or "").strip()

        if resp.usage is not None:
            self.stats.prompt_tokens += resp.usage.prompt_tokens or 0
            self.stats.completion_tokens += resp.usage.completion_tokens or 0
        if choice.finish_reason == "length":
            self.stats.truncated += 1

        return content, message, choice.finish_reason

    # ------------------------------------------------------------------ public
    def complete(self, prompt, system=None, max_tokens=None, temperature=None):
        max_tokens = self.default_max_tokens if max_tokens is None else int(max_tokens)
        temperature = self.default_temperature if temperature is None else float(temperature)
        key = self._key(prompt, system, max_tokens, temperature)

        # ---- cache / single-flight
        while True:
            with self._lock:
                if key in self._cache:
                    self.stats.cache_hits += 1
                    return self._cache[key]
                event = self._inflight.get(key)
                if event is None:
                    event = threading.Event()
                    self._inflight[key] = event
                    break
            event.wait(timeout=max(60.0, REQUEST_TIMEOUT_S * 2))

        try:
            last_error = None
            for attempt in range(3):
                try:
                    content, message, finish = self._call_once(
                        prompt, system, max_tokens, temperature)

                    if content:
                        with self._lock:
                            self._cache[key] = content
                        return content

                    # ---- empty content: is this a reasoning model?
                    self.stats.empty_outputs += 1
                    reasoning = (getattr(message, "reasoning", None)
                                 or getattr(message, "reasoning_content", None) or "")
                    if (reasoning and self.disable_reasoning == "auto"
                            and not self._extra_body):
                        # Spend the budget on hidden tokens -> disable thinking once.
                        self._extra_body = {"reasoning_effort": "none"}
                        self.stats.reasoning_fallbacks += 1
                        continue
                    if reasoning and finish == "length":
                        return ""      # truncated think-only answer: caller records a miss
                    return ""

                except Exception as exc:
                    last_error = exc
                    status = getattr(exc, "status_code", None)
                    if status is None:
                        status = getattr(getattr(exc, "response", None), "status_code", None)
                    text = str(exc).lower()
                    transient = (
                        status in (408, 409, 429, 500, 502, 503, 504)
                        or any(tok in text for tok in (
                            "timeout", "timed out", "temporarily", "connection reset",
                            "rate limit", "overloaded", "connection refused"))
                    )
                    permanent = any(tok in text for tok in (
                        "unauthorized", "authentication", "model not found",
                        "not found", "invalid api key"))
                    if permanent or not transient or attempt == 2:
                        raise RuntimeError(
                            f"{self.role} LLM call failed ({self.model}): {exc}") from exc
                    self.stats.retries += 1
                    delay = (2 ** attempt) + self._jitter.random()
                    time.sleep(delay)

            raise RuntimeError(
                f"{self.role} LLM call failed ({self.model}): {last_error}") from last_error
        finally:
            with self._lock:
                self._inflight.pop(key, None)
            event.set()


optimizer_client = LLMClient(
    OPTIMIZER_MODEL, "optimizer",
    default_max_tokens=OPTIMIZER_MAX_TOKENS,
    default_temperature=0.8,
)
target_client = LLMClient(
    TARGET_MODEL, "target",
    default_max_tokens=TARGET_MAX_TOKENS,
    default_temperature=TARGET_TEMPERATURE,
)

# ---- connectivity + reasoning-behaviour probe (one request per role)
for _client in (optimizer_client, target_client):
    _probe = _client.complete(
        "Reply with the single token OK and nothing else.", max_tokens=32, temperature=0.0)
    _again = _client.complete(
        "Reply with the single token OK and nothing else.", max_tokens=32, temperature=0.0)
    print(f"{_client.role:9s} {_client.model:14s} probe={_probe!r:12s} "
          f"cache_hits={_client.stats.cache_hits} empty={_client.stats.empty_outputs} "
          f"thinking_forced_off={_client.stats.reasoning_fallbacks}")
    if not _probe:
        print(f"  WARNING: {_client.model} returned an empty completion. "
              f"Scores will be 0; check the reasoning/token settings.")
    assert _client.stats.cache_hits >= 1, "request cache is not working"

## 6. Generation strategies

The five strategies of paper §2.3.1.2, implemented as *generators* only (the paper unifies
evaluation and selection across methods):

| Strategy | Weight | Behaviour |
|---|---|---|
| `random_mutation` | 0.40 | one of the 8 paper mutation types, chosen uniformly |
| `apo` | 0.20 | textual gradient from the parent's misclassified examples |
| `opro` | 0.20 | new prompt from the ranked prompt trajectory (no dropout, as in the paper) |
| `fewshot` | 0.10 | append 1–3 labelled **training** examples |
| `crossover` | 0.10 | word-midpoint splice of two parents, then an LLM cleanup |

Changes from the original implementation, all aimed at *measured* correctness:

* **APO errors come from the training set** by default (`APO_ERROR_SOURCE="train"`), matching
  §2.3.3.1. The train split is scored once per unique parent and cached, so the extra cost is
  bounded (~50 target calls per new parent).
* **Every generated candidate is validated**: code fences are stripped, `{input}` is restored, and
  the eight category names are checked. If a candidate cannot be repaired it is rejected and
  retried up to `MAX_GENERATION_ATTEMPTS` times. The original accepted whatever the optimizer
  returned, which is how a ```json-fenced object missing `directed_vs_generalized` became the
  reported "best prompt".
* **No silent cross-strategy fallback.** If a strategy lacks its context it returns `None` and is
  logged as `*_rejected`; it never borrows `random_mutator`, which would contaminate the
  `without_random_mutation` ablation.
* Few-shot sampling uses the run's `random.Random` instance instead of a global `randint`, so it is
  reproducible and does not leak state between runs.

In [ ]:
import json
import random

MUTATION_TYPES = {
    "instruction_expansion": "Expand the instructions with more explicit detail about how to decide each category.",
    "expert_persona_injection": "Add a persona at the start, e.g. 'You are an expert hate-speech researcher.'",
    "structural_variation": "Restructure the prompt (e.g. reorder sections, use numbered steps) while keeping the same meaning.",
    "constraint_addition": "Add a stricter output-format constraint so the model cannot add any extra text.",
    "creative_backstory": "Add a short scenario/backstory framing why this classification matters.",
    "task_decomposition": "Break the task into explicit sub-steps (consider each category individually, then output the final JSON array).",
    "concise_optimization": "Make the prompt more concise while preserving all required instructions.",
    "role_assignment": "Explicitly assign the model a role/responsibility (e.g. 'Your sole job is to output valid JSON labels.').",
}

PROMPT_CONTRACT = (
    "The template MUST keep the literal placeholder {input} somewhere in it, and MUST still "
    "instruct the model to output ONLY a JSON array of category names."
)


def validate_prompt(prompt):
    # True iff the prompt is usable by every strategy and still names all categories.
    if not prompt or "{input}" not in prompt:
        return False
    low = prompt.lower()
    return all(lbl in low for lbl in LABELS)


def _ensure_placeholder(prompt):
    if "{input}" not in prompt:
        prompt = prompt.rstrip() + "\n\nMessage: {input}"
    return prompt


def finalize_candidate(raw_text):
    # Repair a generated prompt, or return None if it cannot be made valid.
    if not raw_text or not raw_text.strip():
        return None
    text = _ensure_placeholder(strip_code_fences(raw_text))
    missing = [lbl for lbl in LABELS if lbl.lower() not in text.lower()]
    if missing:
        text = text.rstrip() + "\n\nCategories to choose from: " + ", ".join(LABELS) + "."
    return text if validate_prompt(text) else None


def random_mutator(parent_prompt, optimizer_client, rng):
    kind, instruction = rng.choice(list(MUTATION_TYPES.items()))
    meta_prompt = (
        "You are editing a prompt template used for multilabel text classification.\n"
        f"{PROMPT_CONTRACT}\n\n"
        f"Current prompt template:\n---\n{parent_prompt}\n---\n\n"
        f"Apply this edit ({kind}): {instruction}\n"
        "Return ONLY the new prompt template, nothing else."
    )
    return optimizer_client.complete(meta_prompt, temperature=0.8), kind


def apo_strategy(parent_prompt, errors, optimizer_client, n_errors=5):
    lines = []
    for err in errors[:n_errors]:
        lines.append(
            f"- Message: {err['input'][:160]!r}\n"
            f"  Gold labels: {err['gold']}\n"
            f"  Model predicted: {err['pred']}"
        )
    error_text = "\n".join(lines)
    meta_prompt = (
        "You are improving a prompt template for multilabel hate-speech classification. "
        "Below are examples the current prompt got WRONG:\n\n"
        f"{error_text}\n\n"
        f"Current prompt template:\n---\n{parent_prompt}\n---\n\n"
        "Analyse why these errors happen, then rewrite the prompt template to fix them.\n"
        f"{PROMPT_CONTRACT}\n"
        "Return ONLY the new prompt template."
    )
    return optimizer_client.complete(meta_prompt, temperature=0.7)


def opro_strategy(trajectory, optimizer_client, k=5):
    # trajectory is sorted best-first; the paper's GAAPO uses no trajectory dropout.
    lines = []
    for i, (prompt, score) in enumerate(trajectory[:k]):
        lines.append(f"{i + 1}. score={score:.3f}\n{prompt}\n")
    meta_prompt = (
        "You are optimising a prompt template for multilabel hate-speech classification. "
        "Below is a ranked history of previous prompts and their validation scores "
        "(higher is better, exact-match accuracy):\n\n"
        + "\n".join(lines) +
        "\nPropose a NEW prompt template likely to score higher than all of the above, by "
        "learning from what worked.\n"
        f"{PROMPT_CONTRACT}\n"
        "Return ONLY the new prompt template."
    )
    return optimizer_client.complete(meta_prompt, temperature=0.8)


def fewshot_strategy(parent_prompt, train_df, rng):
    k = min(rng.choice([1, 2, 3]), len(train_df))
    examples = train_df.iloc[rng.sample(range(len(train_df)), k)]
    chunks = [
        f"Example message: {row['input']}\nExample labels: {json.dumps(row['labels'])}"
        for _, row in examples.iterrows()
    ]
    return parent_prompt + "\n\nHere are some labelled examples:\n" + "\n\n".join(chunks)


def crossover_strategy(parent1, parent2, optimizer_client, cleanup=True):
    w1, w2 = parent1.split(), parent2.split()
    if len(w1) < 2 or len(w2) < 2:
        return None
    child = _ensure_placeholder(" ".join(w1[: len(w1) // 2] + w2[len(w2) // 2:]))
    if cleanup:
        meta_prompt = (
            "The following prompt template was created by splicing together two other prompt "
            "templates and may be incoherent. Rewrite it so it reads clearly.\n"
            f"{PROMPT_CONTRACT}\n"
            f"Return ONLY the cleaned-up prompt.\n\nSpliced prompt:\n---\n{child}\n---"
        )
        child = optimizer_client.complete(meta_prompt, temperature=0.6)
    return child


# ---- cheap offline checks
assert finalize_candidate("") is None
assert validate_prompt(BASELINE_PROMPT)
_fixed = finalize_candidate("```json\n{\"categories\": [\"race\"]}\n```")
assert validate_prompt(_fixed) and "{input}" in _fixed
print("strategy helpers self-check passed")

## 7. Population allocation and dispatch

Paper §2.3.3.2: *"The number of candidates per strategy is calculated by multiplying these weights
by the total population size. To maintain the exact desired population size, any remaining slots are
allocated to the strategy with the highest weight."*

`allocate_candidates` implements that deterministically. For the paper's weights
(40/20/20/10/10) and any population that is a multiple of 10 the quotas are integral, so this is
exactly the paper's rule. When weights are renormalised (the ablation) the quotas are not integral;
we then use largest-remainder apportionment, which coincides with the paper's "give the remainder
to the highest weight" whenever there is a single leftover slot and is well defined otherwise. The
original drew each slot independently at random, which produced e.g. 47 % / 19 % / 19 % / 8 % / 8 %
instead of 40/20/20/10/10 and added variance between runs and between ablation configs.

`generate_candidate` takes an explicit `GenerationContext` (parents, trajectory, caches, RNG)
instead of reading globals, so a run cannot leak state into the next one.

In [ ]:
import math
from collections import Counter
from dataclasses import dataclass

STRATEGY_WEIGHTS = {
    "random_mutation": 0.4,
    "apo": 0.2,
    "opro": 0.2,
    "fewshot": 0.1,
    "crossover": 0.1,
}
STRATEGY_ORDER = list(STRATEGY_WEIGHTS)


def allocate_candidates(weights, n, order=None, floor=0):
    # Deterministic per-generation strategy quotas (paper Sec. 2.3.3.2).
    # floor=1 guarantees every strategy at least one slot, so an operator can
    # never go extinct; used by the adaptive and random arms.
    order = list(order or [k for k in STRATEGY_ORDER if k in weights])
    if n <= 0:
        return []
    total = sum(max(0.0, weights.get(k, 0.0)) for k in order)
    if total <= 0:
        raise ValueError("All strategy weights are zero")
    raw = {k: n * max(0.0, weights.get(k, 0.0)) / total for k in order}
    base = {k: int(math.floor(raw[k])) for k in order}
    remainder = n - sum(base.values())
    ranked = sorted(order, key=lambda k: (-(raw[k] - base[k]), order.index(k)))
    for k in ranked[:max(0, remainder)]:
        base[k] += 1
    if floor and n >= floor * len(order):
        for k in order:
            while base[k] < floor:
                donor = max(order, key=lambda j: base[j])
                if base[donor] <= floor:
                    break
                base[donor] -= 1
                base[k] += 1
    out = []
    for k in order:
        out.extend([k] * base[k])
    assert len(out) == n, (len(out), n)
    return out


def pick_strategy(rng, weights=None):
    # Kept for STRATEGY_ALLOCATION="random" and for comparisons with the original notebook.
    w = weights or STRATEGY_WEIGHTS
    names = [k for k in STRATEGY_ORDER if w.get(k, 0.0) > 0]
    return rng.choices(names, weights=[w[k] for k in names], k=1)[0]


def renormalized_weights(weights, exclude):
    remaining = {k: v for k, v in weights.items() if k != exclude}
    total = sum(remaining.values())
    if total <= 0:
        raise ValueError(f"Cannot exclude {exclude}: no weight left")
    return {k: v / total for k, v in remaining.items()}


@dataclass
class GenerationContext:
    parents: list
    parent_ids: list
    trajectory: list
    train_errors: dict
    val_errors: dict
    train_df: object
    optimizer_client: object
    target_client: object
    rng: object
    apo_error_source: str = "train"
    crossover_cleanup: bool = True


def generate_candidate(strategy, ctx):
    # Returns (prompt_or_None, strategy_label, parent_ids, note).
    i = ctx.rng.randrange(len(ctx.parents))
    parent, parent_id = ctx.parents[i], ctx.parent_ids[i]

    if strategy == "random_mutation":
        raw, kind = random_mutator(parent, ctx.optimizer_client, ctx.rng)
        return finalize_candidate(raw), f"random_mutation:{kind}", [parent_id], None

    if strategy == "apo":
        if ctx.apo_error_source == "train":
            if parent not in ctx.train_errors:
                # Paper Sec. 2.3.3.1: APO analyses errors on the TRAIN split.
                ctx.train_errors[parent] = evaluate_on_examples(
                    parent, ctx.train_df, ctx.target_client).errors
            errors = ctx.train_errors[parent]
        else:
            errors = ctx.val_errors.get(parent, [])
        if not errors:
            return None, "apo", [parent_id], "no_errors"
        return finalize_candidate(
            apo_strategy(parent, errors, ctx.optimizer_client)), "apo", [parent_id], None

    if strategy == "opro":
        if not ctx.trajectory:
            return None, "opro", [parent_id], "empty_trajectory"
        return finalize_candidate(
            opro_strategy(ctx.trajectory, ctx.optimizer_client)), "opro", [parent_id], None

    if strategy == "fewshot":
        return finalize_candidate(
            fewshot_strategy(parent, ctx.train_df, ctx.rng)), "fewshot", [parent_id], None

    if strategy == "crossover":
        if len(ctx.parents) < 2:
            return None, "crossover", [parent_id], "needs_two_parents"
        i1, i2 = ctx.rng.sample(range(len(ctx.parents)), 2)
        raw = crossover_strategy(ctx.parents[i1], ctx.parents[i2],
                                 ctx.optimizer_client, cleanup=ctx.crossover_cleanup)
        return finalize_candidate(raw), "crossover", [ctx.parent_ids[i1], ctx.parent_ids[i2]], None

    raise ValueError(f"Unknown strategy: {strategy}")


# ---- allocation self-checks
assert Counter(allocate_candidates(STRATEGY_WEIGHTS, 10)) == \
       Counter({"random_mutation": 4, "apo": 2, "opro": 2, "fewshot": 1, "crossover": 1})
assert Counter(allocate_candidates(STRATEGY_WEIGHTS, 50)) == \
       Counter({"random_mutation": 20, "apo": 10, "opro": 10, "fewshot": 5, "crossover": 5})
_abl = allocate_candidates(renormalized_weights(STRATEGY_WEIGHTS, "random_mutation"), 10)
assert len(_abl) == 10 and "random_mutation" not in _abl
_skew = {k: 0.01 for k in STRATEGY_WEIGHTS}
_skew["random_mutation"] = 0.96
_floor_check = Counter(allocate_candidates(_skew, 10, floor=1))
assert len(_floor_check) == 5 and min(_floor_check.values()) >= 1, _floor_check
print("allocation self-checks passed:", Counter(allocate_candidates(STRATEGY_WEIGHTS, 20)))

## 8. Evaluation

Three selection strategies, all returning the same `EvalResult` object so the loop is uniform:

* **`all`** — every unique candidate on the full 50-example validation set (paper's *complete
  evaluation*). Parallelised over every `(candidate, example)` pair.
* **`successive_halving`** — chunks of 20 % of the validation set, rank on the **current** chunk,
  keep the top 60 % (i.e. eliminate 40 %), repeat. The original ranked candidates on a *cumulative*
  mean, so the first ~4 examples decided every later round; that is the volatility the paper warns
  about, not a property of SH.
* **`bandit`** — UCB-E style exploration with index `mean + sqrt(a / pulls)`,
  `a = (T - K) / K`. The original used a UCB1-style bonus and re-used examples through
  `idx % len(pool)` while reporting the pull count as if it were a sample size.

`score_kind` records which of the two kinds of number you are looking at. Subset-based scores are
fine for *ranking* inside a generation but they are **not** comparable with full-validation scores,
so in `successive_halving`/`bandit` mode only the generation winner is re-scored on the full
validation set and only that score can enter the hall of fame. `n_unique_examples` is reported
separately from `n` so repeated bandit pulls cannot masquerade as extra data.

In [ ]:
import concurrent.futures
from dataclasses import dataclass, field


@dataclass
class EvalResult:
    score: float
    n: int
    errors: list
    macro_f1: float
    parse_methods: dict
    n_parse_failures: int
    score_kind: str = "full_val"
    n_unique_examples: int = 0
    predictions: list = field(default_factory=list)
    golds: list = field(default_factory=list)


def _prompt_for(template, text):
    if "{input}" not in template:
        raise ValueError("Prompt template is missing the {input} placeholder")
    return template.replace("{input}", text)


def _records_of(df):
    return df[["input", "labels"]].to_dict("records")


def _classify(prompt, record, client):
    raw = client.complete(_prompt_for(prompt, record["input"]),
                          max_tokens=TARGET_MAX_TOKENS, temperature=TARGET_TEMPERATURE)
    pred, method = parse_prediction(raw)
    return pred, set(record["labels"]), method


def _result_from(records, outcomes, score_kind="full_val", n_unique=None):
    n = len(outcomes)
    preds = [p for p, _, _ in outcomes]
    golds = [g for _, g, _ in outcomes]
    correct = sum(1 for p, g, _ in outcomes if p == g)
    methods = Counter(m for _, _, m in outcomes)
    return EvalResult(
        score=(correct / n) if n else 0.0,
        n=n,
        errors=[{"input": rec["input"], "gold": sorted(g), "pred": sorted(p)}
                for rec, (p, g, _) in zip(records, outcomes) if p != g],
        macro_f1=macro_f1(preds, golds),
        parse_methods=dict(methods),
        n_parse_failures=methods.get("empty", 0),
        score_kind=score_kind,
        n_unique_examples=n if n_unique is None else n_unique,
        predictions=[sorted(p) for p in preds],
        golds=[sorted(g) for g in golds],
    )


def _pool(max_workers):
    return max(1, int(max_workers or MAX_CONCURRENT_REQUESTS))


def evaluate_on_examples(prompt, examples_df, client, max_workers=None):
    # Strict set-equality accuracy of one prompt on one dataframe.
    records = _records_of(examples_df)
    if not records:
        return EvalResult(0.0, 0, [], float("nan"), {}, 0)
    with concurrent.futures.ThreadPoolExecutor(max_workers=_pool(max_workers)) as ex:
        outcomes = list(ex.map(lambda rec: _classify(prompt, rec, client), records))
    return _result_from(records, outcomes)


def evaluate_all_candidates(candidates, val_df, client, max_workers=None):
    # Shared pool over every (unique candidate, example) pair. Keyed by position, not by
    # prompt string, so identical prompts can never merge their evaluations.
    candidates = list(dict.fromkeys(candidates))
    records = _records_of(val_df)
    tasks = [(ci, ri) for ci in range(len(candidates)) for ri in range(len(records))]
    grouped = [[] for _ in candidates]

    def one(task):
        ci, ri = task
        return ci, _classify(candidates[ci], records[ri], client)

    with concurrent.futures.ThreadPoolExecutor(max_workers=_pool(max_workers)) as ex:
        for ci, outcome in ex.map(one, tasks):
            grouped[ci].append(outcome)
    return {c: _result_from(records, grouped[i]) for i, c in enumerate(candidates)}


def successive_halving_eval(candidates, val_df, client, chunk_frac=0.2,
                            eliminate_frac=0.4, rng=None, max_workers=None):
    rng = rng or random.Random(RANDOM_SEED)
    candidates = list(dict.fromkeys(candidates))
    pool = val_df.sample(frac=1.0, random_state=rng.randrange(2 ** 31)).reset_index(drop=True)
    chunk_size = max(1, int(round(len(pool) * chunk_frac)))

    remaining = list(candidates)
    correct = {c: 0 for c in candidates}
    seen = {c: 0 for c in candidates}
    errors = {c: [] for c in candidates}
    methods = {c: Counter() for c in candidates}
    ptr, round_no = 0, 0

    while len(remaining) > 1 and ptr < len(pool):
        round_no += 1
        chunk = pool.iloc[ptr:ptr + chunk_size]
        ptr += len(chunk)
        if chunk.empty:
            break
        round_score = {}
        for c in remaining:
            res = evaluate_on_examples(c, chunk, client, max_workers=max_workers)
            round_score[c] = res.score
            correct[c] += int(round(res.score * res.n))
            seen[c] += res.n
            errors[c].extend(res.errors)
            methods[c].update(res.parse_methods)
        # Paper: compare on the CURRENT subset.
        ranked = sorted(remaining, key=lambda c: (-round_score[c], candidates.index(c)))
        keep = min(max(1, int(math.ceil(len(ranked) * (1.0 - eliminate_frac)))),
                   len(ranked) - 1)
        remaining = ranked[:keep]
        print(f"    [successive_halving round {round_no}] chunk={len(chunk)} "
              f"kept={len(remaining)}/{len(candidates)}")

    return {
        c: EvalResult(
            score=(correct[c] / seen[c]) if seen[c] else 0.0,
            n=seen[c], errors=errors[c], macro_f1=float("nan"),
            parse_methods=dict(methods[c]), n_parse_failures=methods[c].get("empty", 0),
            score_kind="subset", n_unique_examples=seen[c],
        ) for c in candidates
    }


def bandit_eval(candidates, val_df, client, pulls_per_candidate=10, rng=None):
    rng = rng or random.Random(RANDOM_SEED)
    candidates = list(dict.fromkeys(candidates))
    records = _records_of(val_df)
    K, M = len(candidates), len(records)
    if M == 0:
        return {c: EvalResult(0.0, 0, [], float("nan"), {}, 0, "subset", 0) for c in candidates}
    T = max(K, int(pulls_per_candidate) * K)
    a = (T - K) / K if K > 1 else 1.0

    pulls = {c: 0 for c in candidates}
    wins = {c: 0 for c in candidates}
    errors = {c: [] for c in candidates}
    methods = {c: Counter() for c in candidates}
    touched = {c: set() for c in candidates}

    def pull(c, idx):
        record = records[idx]
        pred, gold, method = _classify(c, record, client)
        pulls[c] += 1
        touched[c].add(idx)
        methods[c][method] += 1
        wins[c] += 1 if pred == gold else 0
        if pred != gold:
            errors[c].append({"input": record["input"], "gold": sorted(gold),
                              "pred": sorted(pred)})

    for c in candidates:                       # one initial pull per arm
        pull(c, rng.randrange(M))
    next_idx = 0
    while sum(pulls.values()) < T:
        c = max(candidates, key=lambda k: (wins[k] / pulls[k]) + math.sqrt(a / pulls[k]))
        pull(c, next_idx % M)
        next_idx += 1

    return {
        c: EvalResult(
            score=wins[c] / pulls[c], n=pulls[c], errors=errors[c], macro_f1=float("nan"),
            parse_methods=dict(methods[c]), n_parse_failures=methods[c].get("empty", 0),
            score_kind="subset", n_unique_examples=len(touched[c]),
        ) for c in candidates
    }

## 9. The GAAPO loop

Generation 0 evaluates the seed prompt on the **same** validation set that candidates will be
scored on (the original scored the seed on 10 examples and then reported "0.00 baseline vs 0.32
best" against candidates scored on 50).

Per generation: allocate quotas → shuffle the order → generate (validating and retrying each
candidate) → collapse identical candidates → evaluate → log → promote the top
`max(2, population_size // 4)` prompts from the *deduplicated* global trajectory as parents
(paper §2.3.1.1: "choose, among all the prompts which have been evaluated, the best").

`run_gaapo` holds **all** of its state locally and returns it, so successive runs cannot contaminate
each other. The original mutated module-level `hall_of_fame`, `candidate_log`, `STRATEGY_WEIGHTS`
and `prompt_ids`, which is why the ablation needed a global reset and why `parent_ids` were not
comparable between configs.

In [ ]:
import itertools
import time
from concurrent.futures import ThreadPoolExecutor


def _slot_rng(seed, generation, slot):
    # Deterministic per-slot stream: makes parallel generation order-independent.
    return random.Random((int(seed) * 1_000_003 + generation * 1009 + slot) ^ 0x5EED)


def run_gaapo(train_df, val_df, population_size, generations, eval_mode,
              optimizer_client, target_client, baseline_prompt=None,
              strategy_weights=None, seed=RANDOM_SEED, apo_error_source=None,
              crossover_cleanup=None, weight_controller=None, verbose=True):
    baseline_prompt = baseline_prompt or BASELINE_PROMPT
    apo_error_source = apo_error_source or APO_ERROR_SOURCE
    crossover_cleanup = CROSSOVER_CLEANUP if crossover_cleanup is None else crossover_cleanup
    weights = dict(strategy_weights or STRATEGY_WEIGHTS)
    rng = random.Random(seed)

    t_start = time.time()
    optim0 = optimizer_client.stats.calls
    targ0 = target_client.stats.calls

    prompt_ids, id_counter = {}, itertools.count(1)

    def get_id(prompt):
        if prompt not in prompt_ids:
            prompt_ids[prompt] = next(id_counter)
        return prompt_ids[prompt]

    hall_of_fame = {"best": None, "history": []}
    candidate_log, trajectory = [], []
    val_errors, train_errors = {}, {}
    weights_used = dict(weights)

    def record(result, generation, strategy, prompt, parent_ids, slot_count, calls_delta):
        entry = {"prompt": prompt, "score": result.score, "generation": generation,
                 "strategy": strategy, "n_examples": result.n,
                 "score_kind": result.score_kind}
        hall_of_fame["history"].append(entry)
        if hall_of_fame["best"] is None or result.score > hall_of_fame["best"]["score"]:
            hall_of_fame["best"] = entry
        candidate_log.append({
            "generation": generation,
            "strategy": strategy,
            "parent_ids": ";".join(str(p) for p in parent_ids),
            "prompt": prompt,
            "validation_score": result.score,
            "n_examples": result.n,
            "n_unique_examples": result.n_unique_examples,
            "score_kind": result.score_kind,
            "macro_f1": result.macro_f1,
            "parse_methods": json.dumps(result.parse_methods),
            "n_parse_failures": result.n_parse_failures,
            "slot_count": slot_count,
            "parent_score": None,
            "weights_json": json.dumps(weights_used),
            "requests_this_gen": calls_delta,
            "requests_total": (optimizer_client.stats.calls + target_client.stats.calls),
        })

    # ---- generation 0: the paper's initial full validation evaluation of the seed
    seed_res = evaluate_on_examples(baseline_prompt, val_df, target_client)
    val_errors[baseline_prompt] = seed_res.errors
    trajectory.append((baseline_prompt, seed_res.score))
    get_id(baseline_prompt)
    record(seed_res, 0, "seed", baseline_prompt, [],
           1, optimizer_client.stats.calls + target_client.stats.calls)
    parents, parent_ids = [baseline_prompt], [get_id(baseline_prompt)]
    if verbose:
        print(f"Gen 0 (seed): val={seed_res.score:.3f} on {seed_res.n} examples "
              f"(macro-F1 {seed_res.macro_f1:.3f})")

    for generation in range(1, generations + 1):
        gen_t0 = time.time()
        calls0 = optimizer_client.stats.calls + target_client.stats.calls

        # ---- population allocation
        if weight_controller is not None:
            weights_used = weight_controller.next_weights()
            strategies = allocate_candidates(weights_used, population_size, floor=1)
        elif STRATEGY_ALLOCATION == "quota":
            weights_used = dict(weights)
            strategies = allocate_candidates(weights, population_size)
        else:
            weights_used = dict(weights)
            strategies = [pick_strategy(rng, weights) for _ in range(population_size)]
        rng.shuffle(strategies)

        # ---- APO reads the target model's errors on the TRAIN split. Compute them
        # once per parent so the (parallel) generation phase is read-only.
        if apo_error_source == "train" and "apo" in strategies:
            for _parent in parents:
                if _parent not in train_errors:
                    train_errors[_parent] = evaluate_on_examples(
                        _parent, train_df, target_client).errors

        def _generate(item):
            slot, strategy = item
            local_ctx = GenerationContext(
                parents=list(parents), parent_ids=list(parent_ids),
                trajectory=sorted(trajectory, key=lambda t: -t[1])[:10],
                train_errors=train_errors, val_errors=val_errors, train_df=train_df,
                optimizer_client=optimizer_client, target_client=target_client,
                rng=_slot_rng(seed, generation, slot),
                apo_error_source=apo_error_source, crossover_cleanup=crossover_cleanup,
            )
            failed = 0
            for _ in range(MAX_GENERATION_ATTEMPTS):
                prompt, label, used_ids, _note = generate_candidate(strategy, local_ctx)
                if prompt is not None and validate_prompt(prompt):
                    return prompt, label, used_ids, failed
                failed += 1
            return None, strategy, [], failed

        # ---- generate the population concurrently (per-slot RNG keeps this deterministic)
        with ThreadPoolExecutor(max_workers=max(1, MAX_CONCURRENT_REQUESTS)) as pool:
            raw_generated = list(pool.map(_generate, list(enumerate(strategies))))

        generated, rejections = [], 0
        for prompt, label, used_ids, failed in raw_generated:
            rejections += failed
            if prompt is None:
                # Do not borrow another strategy: fall back to a parent, which the
                # deduplication below collapses into an existing individual.
                prompt, label, used_ids = parents[0], f"{label}_rejected", [parent_ids[0]]
            generated.append((prompt, label, used_ids))

        slot_count = Counter(p for p, _, _ in generated)
        meta, unique = {}, []
        for prompt, label, used_ids in generated:
            if prompt not in meta:
                meta[prompt] = (label, used_ids)
                unique.append(prompt)

        if eval_mode == "all":
            results = evaluate_all_candidates(unique, val_df, target_client)
        elif eval_mode == "successive_halving":
            results = successive_halving_eval(unique, val_df, target_client, rng=rng)
        elif eval_mode == "bandit":
            results = bandit_eval(unique, val_df, target_client, rng=rng)
        else:
            raise ValueError(f"Unknown eval_mode: {eval_mode}")

        winner = max(unique, key=lambda c: (results[c].score, -unique.index(c)))
        if eval_mode != "all":
            # Make the winner's score comparable with full-validation scores.
            results[winner] = evaluate_on_examples(winner, val_df, target_client)

        for prompt in unique:
            result = results[prompt]
            label, used_ids = meta[prompt]
            val_errors[prompt] = result.errors
            trajectory.append((prompt, result.score))
            get_id(prompt)
            record(result, generation, label, prompt, used_ids, slot_count[prompt],
                   optimizer_client.stats.calls + target_client.stats.calls - calls0)

        # ---- credit assignment for the adaptive controller
        if weight_controller is not None:
            id_to_score = {prompt_ids[p]: s for p, s in trajectory}
            observations = []
            for prompt, label, used_ids in generated:
                base = label.split(":")[0]
                rejected = base.endswith("_rejected")
                if rejected:
                    base = base[: -len("_rejected")]
                observations.append({
                    "strategy": base,
                    "score": results[prompt].score,
                    "parent_scores": [id_to_score[i] for i in used_ids if i in id_to_score],
                    "rejected": rejected,
                })
            weight_controller.observe(observations)

        # ---- selection: global best evaluated prompts become the parents
        ranked = sorted(dict(trajectory).items(), key=lambda kv: -kv[1])
        top_n = max(1, min(len(ranked), max(2, population_size // 4)))
        parents = [p for p, _ in ranked[:top_n]]
        parent_ids = [get_id(p) for p in parents]

        if verbose:
            print(f"Gen {generation}: best={results[winner].score:.3f}  "
                  f"unique={len(unique)}/{population_size}  "
                  f"rejected={rejections}  dupes={sum(v - 1 for v in slot_count.values())}  "
                  f"HoF={hall_of_fame['best']['score']:.3f}  "
                  f"{time.time() - gen_t0:.1f}s")
            if weight_controller is not None:
                print("    weights: " + "  ".join(
                    f"{k}={v:.2f}" for k, v in weights_used.items()))

    meta_out = {
        "seed": seed,
        "population_size": population_size,
        "generations": generations,
        "eval_mode": eval_mode,
        "strategy_weights": weights,
        "weight_controller": getattr(weight_controller, "name", None),
        "apo_error_source": apo_error_source,
        "crossover_cleanup": crossover_cleanup,
        "strategy_allocation": STRATEGY_ALLOCATION,
        "n_logged_candidates": len(candidate_log),
        "requests_optimizer": optimizer_client.stats.calls - optim0,
        "requests_target": target_client.stats.calls - targ0,
        "elapsed_s": round(time.time() - t_start, 1),
    }
    meta_out["requests_total"] = meta_out["requests_optimizer"] + meta_out["requests_target"]
    return {"hall_of_fame": hall_of_fame, "candidate_log": candidate_log,
            "meta": meta_out, "prompt_ids": prompt_ids, "trajectory": trajectory,
            "weight_history": list(weight_controller.history) if weight_controller else []}

## 10. Smoke test

A two-generation, 10-candidate run on 6 train / 4 validation / 4 test examples. It exercises all
five strategies, the crossover fallback path, candidate validation, deduplication and the whole
evaluation pipeline for a couple of minutes of local inference — run this before committing to the
overnight sweep.

Invariants asserted (these are exactly the guarantees the original notebook broke):
`n_examples` equals the validation size for **every** row, no `(generation, prompt)` pair appears
twice, and all scores are in `[0, 1]`.

In [ ]:
if RUN_SMOKE_TEST:
    smoke_dir = OUTPUT_DIR / "smoke"
    smoke_dir.mkdir(parents=True, exist_ok=True)

    smoke_run = run_gaapo(
        train_df.head(6), val_df.head(4),
        population_size=10, generations=2, eval_mode="all",
        optimizer_client=optimizer_client, target_client=target_client,
        seed=1234, verbose=True,
    )
    smoke_log = pd.DataFrame(smoke_run["candidate_log"])
    print()
    print(smoke_log[["generation", "strategy", "parent_ids", "validation_score",
                     "n_examples", "slot_count", "n_unique_examples"]].to_string(index=False))

    base_strategies = sorted(smoke_log["strategy"].str.split(":").str[0].unique())
    assert set(smoke_log["n_examples"]) == {4}, smoke_log["n_examples"].unique()
    assert smoke_log.groupby(["generation", "prompt"]).size().max() == 1
    assert smoke_log["validation_score"].between(0.0, 1.0).all()
    assert len(base_strategies) >= 4, base_strategies
    assert smoke_log["n_unique_examples"].max() <= len(val_df.head(4))

    smoke_log.to_csv(smoke_dir / "smoke_candidate_log.csv", index=False)
    print(f"\nSMOKE TEST PASSED in {smoke_run['meta']['elapsed_s']}s "
          f"({smoke_run['meta']['requests_total']} requests) -> {smoke_dir}")
else:
    print("RUN_SMOKE_TEST = False; skipped")

## 11. Main experiment

One GAAPO run at `MAIN_POPULATION_SIZE × MAIN_GENERATIONS` with `MAIN_EVAL_MODE`. Set these in
section 2. The paper's recommended configuration is 50 prompts × 10 generations; its own
budget-matched comparison used 20–50 prompts with 10–25 generations. Anything in that range here is
still a much smaller experiment than the paper's (different models, different split), so treat the
absolute scores as a local reproduction, not a comparison with the published table.

In [ ]:
hall_of_fame, candidate_log, main_run = None, None, None
log_df = None

if RUN_MAIN_EXPERIMENT:
    main_run = run_gaapo(
        train_df, val_df,
        population_size=MAIN_POPULATION_SIZE,
        generations=MAIN_GENERATIONS,
        eval_mode=MAIN_EVAL_MODE,
        optimizer_client=optimizer_client, target_client=target_client,
        seed=RANDOM_SEED, verbose=True,
    )
    hall_of_fame = main_run["hall_of_fame"]
    candidate_log = main_run["candidate_log"]
    log_df = pd.DataFrame(candidate_log)
    print()
    print(f"{main_run['meta']['requests_total']} requests "
          f"(optimizer {main_run['meta']['requests_optimizer']}, "
          f"target {main_run['meta']['requests_target']}) in "
          f"{main_run['meta']['elapsed_s']}s")
    print("best:", {k: v for k, v in hall_of_fame["best"].items() if k != "prompt"})
else:
    print("RUN_MAIN_EXPERIMENT = False; skipped")

## 12. Ablation study — leave-one-out, 3 repeats

Six configurations: the full hybrid mix and each strategy removed (its weight set to 0, the
remaining weights renormalised). Each configuration is run `ABLATION_N_REPEATS` times with the
**same seed per repeat across configurations** (common random numbers), so a configuration
difference cannot be explained by a different random stream.

Three things the original ablation could not support and that are fixed here:

1. **No seeding, one run per config.** The reported spread (0.24–0.44) was within the
   run-to-run noise of a single unlucky/lucky draw. Now every config gets the same seeds and the
   summary reports mean ± std over repeats.
2. **Validation maxima only.** The original ranked configs by the best validation score ever seen,
   which is selection-on-validation over ~170 candidates. Now each run's best prompt is also
   scored on the **held-out test set**, which is the number the paper reports (its Tables 2–4).
3. **Not isolated.** `random_mutation` used to be reachable through the APO/OPRO/crossover
   fallbacks, so `without_random_mutation` could still mutate. Fallbacks now return the parent
   instead of borrowing another strategy.

Caveat to keep in mind when reading the output: scoring *every* configuration on the test set
turns it into a comparison set. The pre-specified comparison is "full hybrid vs each ablation";
do not pick the argmax configuration by test score and call it a finding.

`INCLUDE_SINGLE_STRATEGY = True` adds the paper's actual baselines (APO-only, OPRO-only,
mutator-only, …) which test the stronger claim that the hybrid beats every single strategy. It
costs five more full runs per repeat.

In [ ]:
import time

ablation_df, ablation_runs = None, {}
ablation_test_predictions = None

if RUN_ABLATION:
    ABLATION_CONFIGS = {"full_hybrid": dict(STRATEGY_WEIGHTS)}
    for _strategy in STRATEGY_ORDER:
        ABLATION_CONFIGS[f"without_{_strategy}"] = renormalized_weights(
            STRATEGY_WEIGHTS, _strategy)
    if INCLUDE_SINGLE_STRATEGY:
        for _strategy in STRATEGY_ORDER:
            ABLATION_CONFIGS[f"only_{_strategy}"] = {
                k: (1.0 if k == _strategy else 0.0) for k in STRATEGY_ORDER}

    selected = [c for c in ABLATION_CONFIGS_TO_RUN if c in ABLATION_CONFIGS]
    missing_arms = [c for c in ABLATION_CONFIGS_TO_RUN if c not in ABLATION_CONFIGS]
    if missing_arms:
        print('WARNING unknown ablation arm(s):', missing_arms)
    print(f'running {len(selected)} arms x {ABLATION_N_REPEATS} repeats: {selected}')

    rows, best_prompts, pred_rows = [], {}, []
    for repeat in range(ABLATION_N_REPEATS):
        seed = RANDOM_SEED + 1000 * repeat        # paired across configurations
        for config_name in selected:
            weights = ABLATION_CONFIGS[config_name]
            print(f"\n=== ablation {config_name}  repeat {repeat + 1}/{ABLATION_N_REPEATS} "
                  f"(seed={seed}) ===")
            run = run_gaapo(
                train_df, val_df,
                population_size=ABLATION_POPULATION_SIZE,
                generations=ABLATION_GENERATIONS,
                eval_mode=ABLATION_EVAL_MODE,
                optimizer_client=optimizer_client, target_client=target_client,
                strategy_weights=weights, seed=seed, verbose=True,
            )
            best = run["hall_of_fame"]["best"]
            test_res = evaluate_on_examples(best["prompt"], test_df, target_client)
            ablation_runs[(config_name, repeat)] = run
            rows.append({
                "config": config_name,
                "repeat": repeat,
                "seed": seed,
                "best_val": best["score"],
                "best_val_kind": best.get("score_kind", "full_val"),
                "best_generation": best["generation"],
                "best_strategy": best["strategy"],
                "test_score": test_res.score,
                "test_macro_f1": test_res.macro_f1,
                "n_logged": len(run["candidate_log"]),
                "requests": run["meta"]["requests_total"],
                "elapsed_s": run["meta"]["elapsed_s"],
            })
            best_prompts[f"{config_name}|{repeat}"] = best["prompt"]
            for _i, (_p, _g) in enumerate(zip(test_res.predictions, test_res.golds)):
                pred_rows.append({"config": config_name, "repeat": repeat,
                                  "example": _i, "pred": json.dumps(_p),
                                  "gold": json.dumps(_g)})
            print(f"  -> best_val={best['score']:.3f} (gen {best['generation']}, "
                  f"{best['strategy']})  test={test_res.score:.3f}")
    ablation_df = pd.DataFrame(rows)
    ablation_test_predictions = pd.DataFrame(pred_rows)
    _ablation_dir = OUTPUT_DIR / "ablation"
    _ablation_dir.mkdir(parents=True, exist_ok=True)
    ablation_test_predictions.to_csv(
        _ablation_dir / "ablation_test_predictions.csv", index=False)
    with open(_ablation_dir / "ablation_best_prompts.json", "w") as _fh:
        json.dump(best_prompts, _fh, indent=2)
    print("\nablation finished")
else:
    print("RUN_ABLATION = False; skipped")

In [ ]:
ablation_summary = None

if ablation_df is not None:
    ablation_summary = (ablation_df
                        .groupby("config")
                        .agg(best_val_mean=("best_val", "mean"),
                             best_val_std=("best_val", "std"),
                             test_mean=("test_score", "mean"),
                             test_std=("test_score", "std"),
                             n_repeats=("repeat", "count"))
                        .reset_index())

    if "full_hybrid" not in set(ablation_summary["config"]):
        raise RuntimeError("Ablation is missing the full_hybrid configuration")
    _full = ablation_summary.loc[ablation_summary["config"] == "full_hybrid"].iloc[0]
    ablation_summary["delta_val_vs_full"] = \
        ablation_summary["best_val_mean"] - _full["best_val_mean"]
    ablation_summary["delta_test_vs_full"] = \
        ablation_summary["test_mean"] - _full["test_mean"]

    # Paired (within-repeat) test differences, which is the right error bar for this design.
    piv = ablation_df.pivot_table(index="repeat", columns="config", values="test_score")
    paired = piv.sub(piv["full_hybrid"], axis=0).agg(["mean", "std"]).T
    paired.columns = ["paired_delta_test_mean", "paired_delta_test_std"]

    ablation_summary = ablation_summary.merge(paired, left_on="config", right_index=True,
                                             how="left")
    ablation_summary = ablation_summary.sort_values("test_mean", ascending=False)

    ablation_dir = OUTPUT_DIR / "ablation"
    ablation_dir.mkdir(parents=True, exist_ok=True)
    ablation_df.to_csv(ablation_dir / "ablation_runs.csv", index=False)
    ablation_summary.to_csv(ablation_dir / "ablation_summary.csv", index=False)
    print(ablation_summary.round(3).to_string(index=False))

    _spread = ablation_df.groupby("config")["best_val"].max() - \
              ablation_df.groupby("config")["best_val"].min()
    print(f"\nwithin-config validation spread across repeats: max {_spread.max():.3f}")
    print(f"between-config test spread: "
          f"{ablation_summary['test_mean'].max() - ablation_summary['test_mean'].min():.3f}")
    print("If the within-config spread is comparable to the between-config spread, the "
          "ablation is not resolved — increase repeats or scale before drawing conclusions.")
else:
    print("No ablation results (RUN_ABLATION = False)")

## 13. Selection-method comparison (optional)

Paper §3.7 compared *complete*, *successive halving* and *bandit* selection. Both alternatives were
unreachable and incorrect in the original notebook; here they run the same GAAPO with the same seed
and scale, so the comparison is like-for-like. Off by default because it costs three full runs.

In [ ]:
selection_df = None

if RUN_SELECTION_COMPARISON:
    rows = []
    for mode in ("all", "successive_halving", "bandit"):
        print(f"\n=== selection mode: {mode} ===")
        run = run_gaapo(
            train_df, val_df,
            population_size=MAIN_POPULATION_SIZE, generations=MAIN_GENERATIONS,
            eval_mode=mode, optimizer_client=optimizer_client, target_client=target_client,
            seed=RANDOM_SEED, verbose=True,
        )
        best = run["hall_of_fame"]["best"]
        test_res = evaluate_on_examples(best["prompt"], test_df, target_client)
        rows.append({"eval_mode": mode, "best_val": best["score"], "test_score": test_res.score,
                     "requests": run["meta"]["requests_total"],
                     "elapsed_s": run["meta"]["elapsed_s"]})
    selection_df = pd.DataFrame(rows)
    _base = selection_df.loc[selection_df["eval_mode"] == "all", "requests"].iloc[0]
    selection_df["requests_vs_all_pct"] = 100 * selection_df["requests"] / _base
    selection_dir = OUTPUT_DIR / "selection"
    selection_dir.mkdir(parents=True, exist_ok=True)
    selection_df.to_csv(selection_dir / "selection_methods.csv", index=False)
    print(selection_df.round(3).to_string(index=False))
else:
    print("RUN_SELECTION_COMPARISON = False; skipped")

## 14. Results, held-out test and report

Saves the candidate log, the best prompt, the figures, the ablation outputs and a run manifest, then
scores the best prompt on the 200-example held-out test set.

The baseline and the best prompt are now both scored on the **full validation set**, so the
headline "baseline → best" delta is a like-for-like comparison.

In [ ]:
if hall_of_fame is None:
    print("RUN_MAIN_EXPERIMENT = False: nothing to report")
else:
    log_df = pd.DataFrame(candidate_log)
    log_df.to_csv(OUTPUT_DIR / "candidate_log.csv", index=False)

    best = hall_of_fame["best"]
    with open(OUTPUT_DIR / "best_prompt.json", "w") as fh:
        json.dump(best, fh, indent=2)

    seed_row = log_df[log_df["generation"] == 0].iloc[0]
    baseline_val_score = float(seed_row["validation_score"])
    best_val_score = float(best["score"])
    baseline_test = evaluate_on_examples(BASELINE_PROMPT, test_df, target_client)
    test_res = evaluate_on_examples(best["prompt"], test_df, target_client)

    print(f"baseline prompt      val {baseline_val_score:.3f} "
          f"({int(seed_row['n_examples'])} examples)  test {baseline_test.score:.3f} "
          f"({baseline_test.n} examples)")
    print(f"best evolved prompt  val {best_val_score:.3f} (gen {best['generation']}, "
          f"{best['strategy']})  test {test_res.score:.3f} "
          f"(macro-F1 {test_res.macro_f1:.3f})")
    print(f"delta val  {best_val_score - baseline_val_score:+.3f}   "
          f"delta test {test_res.score - baseline_test.score:+.3f}")
    print(f"test parse methods: {test_res.parse_methods}")
    print(f"\nbest prompt:\n{best['prompt']}")

In [ ]:
import matplotlib.pyplot as plt


def _boxplot(ax, data, tick_labels, **kwargs):
    try:
        ax.boxplot(data, tick_labels=tick_labels, **kwargs)
    except TypeError:                      # matplotlib < 3.9 used `labels`
        ax.boxplot(data, labels=tick_labels, **kwargs)


def add_parent_improvement(frame, prompt_ids):
    # Difference between a candidate's validation score and its best parent's.
    id_to_score = {prompt_ids[p]: s for p, s in zip(frame["prompt"], frame["validation_score"])}
    parents, improvements = [], []
    for _, row in frame.iterrows():
        ids = [int(x) for x in str(row["parent_ids"]).split(";") if x.strip()]
        scores = [id_to_score[i] for i in ids if i in id_to_score]
        parents.append(max(scores) if scores else float("nan"))
        improvements.append(row["validation_score"] - parents[-1] if scores else float("nan"))
    out = frame.copy()
    out["parent_score"] = parents
    out["improvement"] = improvements
    return out


if hall_of_fame is None:
    print("skipped: no main run")
else:
    log_df = add_parent_improvement(log_df, main_run["prompt_ids"])
    log_df.to_csv(OUTPUT_DIR / "candidate_log.csv", index=False)
    fig_dir = OUTPUT_DIR / "figures"
    fig_dir.mkdir(parents=True, exist_ok=True)
    log_df["base_strategy"] = log_df["strategy"].str.split(":").str[0]
    dedup = log_df.drop_duplicates(subset=["prompt"]).copy()

    # 1. best validation score by generation
    per_gen = log_df.groupby("generation")["validation_score"].max()
    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.plot(per_gen.index, per_gen.values, marker="o")
    ax.set_xlabel("Generation")
    ax.set_ylabel("Best validation score")
    ax.set_title("GAAPO: best validation score by generation")
    ax.set_ylim(0, 1.05)
    fig.tight_layout()
    fig.savefig(fig_dir / "score_by_generation.png", dpi=150)
    plt.show()

    # 2. distribution of validation scores per strategy (paper Fig. 6 style)
    stats = (dedup.groupby("base_strategy")["validation_score"]
             .agg(["median", "size"]).sort_values("median", ascending=False))
    tick_labels = [f"{name}\n(n={int(row['size'])})" for name, row in stats.iterrows()]
    data = [dedup.loc[dedup["base_strategy"] == name, "validation_score"].values
            for name in stats.index]
    fig, ax = plt.subplots(figsize=(9, 4.8))
    _boxplot(ax, data, tick_labels, showmeans=True)
    ax.set_ylabel("Validation score")
    ax.set_title("Validation score by generation strategy (unique prompts)")
    fig.tight_layout()
    fig.savefig(fig_dir / "score_by_strategy.png", dpi=150)
    plt.show()

    # 3. mean / max improvement over the parent (paper Fig. 7 style)
    gen_imp = (log_df.dropna(subset=["improvement"])
               .groupby(["generation", "base_strategy"])["improvement"]
               .agg(["mean", "max"]).reset_index())
    fig, ax = plt.subplots(figsize=(9, 5))
    for stat, marker, style in (("mean", ".", "-"), ("max", "x", "--")):
        for strategy, block in gen_imp.groupby("base_strategy"):
            ax.plot(block["generation"], block[stat], marker=marker, linestyle=style,
                    label=f"{strategy} ({stat})", alpha=0.8)
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_xlabel("Generation")
    ax.set_ylabel("Validation improvement vs best parent")
    ax.set_title("Improvement over parent prompts")
    ax.legend(fontsize=7, ncol=2)
    fig.tight_layout()
    fig.savefig(fig_dir / "improvement_vs_parent.png", dpi=150)
    plt.show()

    # 4. parse-method breakdown
    methods = Counter()
    for row in log_df["parse_methods"]:
        methods.update(json.loads(row))
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(list(methods.keys()), list(methods.values()))
    ax.set_ylabel("candidate evaluations")
    ax.set_title("How predictions were parsed")
    fig.tight_layout()
    fig.savefig(fig_dir / "parse_methods.png", dpi=150)
    plt.show()

    print(f"figures written to {fig_dir}")

    if ablation_df is not None:
        summary = ablation_summary.set_index("config").loc[
            [c for c in ablation_summary["config"]]]
        fig, ax = plt.subplots(figsize=(9, 5))
        positions = range(len(summary))
        width = 0.38
        ax.bar([i - width / 2 for i in positions], summary["best_val_mean"], width,
               yerr=summary["best_val_std"].fillna(0), capsize=3, label="validation")
        ax.bar([i + width / 2 for i in positions], summary["test_mean"], width,
               yerr=summary["test_std"].fillna(0), capsize=3, label="test")
        ax.set_xticks(list(positions))
        ax.set_xticklabels(summary.index, rotation=30, ha="right")
        ax.set_ylabel("score (mean +/- std over repeats)")
        ax.set_ylim(0, 1.05)
        ax.set_title("Leave-one-out ablation (paired seeds)")
        ax.legend()
        fig.tight_layout()
        fig.savefig(fig_dir / "ablation_summary.png", dpi=150)
        plt.show()

In [ ]:
import platform
import sys
from datetime import datetime, timezone

if hall_of_fame is None:
    print("RUN_MAIN_EXPERIMENT = False: no manifest written")
else:
    def _to_markdown_table(frame, index=False):
        try:
            return frame.to_markdown(index=index)
        except Exception:
            return frame.to_string(index=index)

    manifest = {
        "generated_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "python": sys.version.split()[0],
        "platform": platform.platform(),
        "optimizer_model": OPTIMIZER_MODEL,
        "target_model": TARGET_MODEL,
        "ollama_base_url": OLLAMA_BASE_URL,
        "random_seed": RANDOM_SEED,
        "config": {
            "apo_error_source": APO_ERROR_SOURCE,
            "strategy_allocation": STRATEGY_ALLOCATION,
            "crossover_cleanup": CROSSOVER_CLEANUP,
            "lenient_parsing": LENIENT_PARSING,
            "max_generation_attempts": MAX_GENERATION_ATTEMPTS,
            "main": {"population": MAIN_POPULATION_SIZE, "generations": MAIN_GENERATIONS,
                     "eval_mode": MAIN_EVAL_MODE},
            "ablation": {"population": ABLATION_POPULATION_SIZE,
                         "generations": ABLATION_GENERATIONS,
                         "eval_mode": ABLATION_EVAL_MODE,
                         "repeats": ABLATION_N_REPEATS,
                         "single_strategy_baselines": INCLUDE_SINGLE_STRATEGY},
        },
        "split": SPLIT_MANIFEST,
        "main_run_meta": main_run["meta"],
        "client_stats": {"optimizer": optimizer_client.stats.as_dict(),
                         "target": target_client.stats.as_dict()},
        "parse_method_counts": dict(PARSE_COUNTS),
        "headline": {
            "baseline_val": baseline_val_score,
            "best_val": best_val_score,
            "baseline_test": baseline_test.score,
            "best_test": test_res.score,
            "best_generation": best["generation"],
            "best_strategy": best["strategy"],
        },
    }
    with open(OUTPUT_DIR / "run_manifest.json", "w") as fh:
        json.dump(manifest, fh, indent=2)

    strategy_table = (log_df.assign(base_strategy=log_df["strategy"].str.split(":").str[0])
                      .groupby("base_strategy")
                      .agg(n_candidates=("prompt", "nunique"),
                           mean_val=("validation_score", "mean"),
                           max_val=("validation_score", "max"))
                      .sort_values("mean_val", ascending=False))

    ablation_block = ("No ablation results." if ablation_df is None
                      else _to_markdown_table(ablation_summary.round(3)))

    report = [
        "# GAAPO experiment report\n",
        f"_generated {manifest['generated_at_utc']} with "
        f"optimizer `{OPTIMIZER_MODEL}` / target `{TARGET_MODEL}`_\n",
        "## Headline\n",
        f"- baseline prompt (full validation): **{baseline_val_score:.3f}**",
        f"- best evolved prompt (full validation): **{best_val_score:.3f}** "
        f"(generation {best['generation']}, strategy `{best['strategy']}`)",
        f"- best evolved prompt (held-out test, n={test_res.n}): **{test_res.score:.3f}** "
        f"(macro-F1 {test_res.macro_f1:.3f})",
        f"- baseline prompt (held-out test): **{baseline_test.score:.3f}**",
        f"- test delta: **{test_res.score - baseline_test.score:+.3f}**\n",
        "Note on the test set: it is used here only to report generalisation. Configurations were "
        "compared on validation; the test number is not used for selection.\n",
        "## Run configuration\n",
        "```json",
        json.dumps(manifest["config"], indent=2),
        "```\n",
        "## Best prompt\n",
        "```",
        best["prompt"],
        "```\n",
        "## Validation score by generation\n",
        _to_markdown_table(log_df.groupby("generation")["validation_score"].max()
                           .to_frame("best_validation_score")),
        "\n\n## Validation score by strategy\n",
        _to_markdown_table(strategy_table.round(3)),
        "\n\n## Parsing breakdown (candidate evaluations)\n",
        _to_markdown_table(pd.Series(dict(PARSE_COUNTS), name="count")
                           .to_frame("count"), index=True),
        "\n\n## Leave-one-out ablation (mean +/- std over "
        f"{ABLATION_N_REPEATS} repeat(s), paired seeds)\n",
        ablation_block,
        "\n\n## Artifacts\n",
        "- `candidate_log.csv`, `best_prompt.json`, `run_manifest.json`",
        "- `figures/score_by_generation.png`, `figures/score_by_strategy.png`, "
        "`figures/improvement_vs_parent.png`, `figures/parse_methods.png`",
        "- `ablation/ablation_runs.csv`, `ablation/ablation_summary.csv`",
        "",
        "_Reminder: these numbers come from a local re-implementation with different models and a "
        "smaller budget than the paper. They are not comparable with the published tables, and no "
        "result in this file should be quoted as reproducing them._",
    ]
    with open(OUTPUT_DIR / "report.md", "w") as fh:
        fh.write("\n".join(report))

    print("wrote:")
    for path in sorted(OUTPUT_DIR.rglob("*")):
        if path.is_file():
            print(" -", path.relative_to(OUTPUT_DIR))

## 15. Change log (original `GAAPO.ipynb` → this file)

**Correctness bugs**

| # | Original behaviour | Impact | Fix |
|---|---|---|---|
| 1 | `run_gaapo` keyed evaluation results by the prompt string, so identical candidates in one generation merged their evaluations | `n_examples` reached 100–300 on a 50-example validation set (verified in the stored `candidate_log.csv`); duplicate rows double-counted in the per-strategy means; `trajectory` filled with duplicates so `parents` could be the same prompt repeated (crossover of a prompt with itself) | Candidates are deduplicated per generation; results are keyed by position; `slot_count` records how many slots produced each prompt |
| 2 | Seed prompt scored on `validation_examples_per_prompt` (=10) examples, candidates on all 50 | The reported "baseline 0.00 → best 0.32" compared different data; a lucky seed on 10 examples could be reported as the evolved prompt | The seed is evaluated on the same validation set as candidates |
| 3 | `successive_halving_eval` ranked on a *cumulative* mean over all chunks seen so far and returned that mean as the final score; kept `max(1, round(0.6·n))` survivors without re-scoring | Not the paper's SH. Survivors' scores depended on how long they survived; ranking was dominated by the first ~4 examples; call counts of different candidates were not comparable | Rank on the current chunk, always eliminate at least one, and report `score_kind="subset"`; only the winner is re-scored on the full validation set and only that score can enter the hall of fame |
| 4 | `bandit_eval` used a UCB1-style bonus `sqrt(1/n)`, re-used examples through `idx % len(pool)`, and reported the pull count as `n_examples` | Budget/`a` not tied to the horizon; repeated examples looked like additional data | UCB-E index with `a = (T-K)/K`; `n` (pulls) and `n_unique_examples` are reported separately |
| 5 | Generated prompts were never validated | The reported best prompt was a code-fenced JSON object that had **lost the `directed_vs_generalized` category** — an invalid prompt that caps achievable accuracy | Fences stripped, `{input}` restored, all eight categories checked, up to `MAX_GENERATION_ATTEMPTS` retries, then rejection |
| 6 | APO/OPRO/crossover fell back to `random_mutator` | `without_random_mutation` could still mutate, so the ablation did not isolate the component it claimed to | Fallbacks return the parent (collapsed as a duplicate) and are logged as `*_rejected`; no cross-strategy borrowing |
| 7 | `local_requests` was cumulative across all ablation runs; `prompt_ids`/`_id_counter` were never reset | Per-config request counts were unrecoverable; `parent_ids` were not comparable across configs | Per-run local state; `requests_this_gen` and per-run deltas in `meta` |
| 8 | `hall_of_fame`, `candidate_log`, `STRATEGY_WEIGHTS` mutated as module globals by name | Fragile; the ablation needed a reset hack and a stale global could silently change results | `run_gaapo` takes weights as an argument and returns all state |
| 9 | `exact_match` defined but unused; the comparison was re-implemented inline | Dead code | `exact_match` is the single metric helper |
| 10 | `ABLATION_VAL_EXAMPLES_PER_PROMPT` did not control the per-candidate validation size in `all` mode (only the seed's) | Misleading name that changed what the baseline measured | Removed; the validation size is explicit |

**Paper fidelity**

| # | Original | Change | Justification |
|---|---|---|---|
| 11 | Weighted random draw per slot | Paper quotas (weight × population, largest-remainder remainder) | Paper §2.3.3.2; the original produced ~47/19/19/8/8 instead of 40/20/20/10/10 |
| 12 | APO used validation errors | `APO_ERROR_SOURCE="train"` by default | Paper §2.3.3.1 assigns error analysis to the train split; using validation for both generation and selection is optimistic |
| 13 | No RNG control | Explicit `random.Random(seed)` per run, paired seeds across ablation configs, seeds recorded in the manifest | The paper repeats experiments 3× and notes LLM non-determinism; without seeds the ablation was uninterpretable |
| 14 | One model for both roles | `OPTIMIZER_MODEL` / `TARGET_MODEL` configured separately | Paper §2.3.4.1 uses distinct optimizer and target LLMs |

**Ablation**

| # | Original | Change |
|---|---|---|
| 15 | 6 configs × 1 unseeded run, compared on the best validation score ever seen | 6 configs × `ABLATION_N_REPEATS` seeded runs (common random numbers), compared on mean ± std of validation **and** held-out test |
| 16 | No test evaluation of ablated configs | Every run's best prompt is scored on the 200-example held-out test set |
| 17 | Markdown said `population_size=10, generations=3`, code used 20/8, stored artifacts came from 10/6 | Scale lives in section 2 and the manifest records exactly what ran |
| 18 | `without_apo` looked best (0.44 vs 0.32) | Reported with error bars and paired deltas; the notebook prints a warning when within-config spread is comparable to between-config spread |

**Reproducibility / hygiene**

| # | Original | Change |
|---|---|---|
| 19 | `MODEL = "phi3.5:latest"` while the title said `qwen2.5:3b`; cell 5 crashed if the model was missing | Two explicit models, availability check that lists what is installed and prints the exact `ollama pull` command |
| 20 | Markdown described Colab + authenticated ngrok; the code used `localhost` and sent no credentials | Documented both paths; `OLLAMA_API_KEY` / `OLLAMA_EXTRA_HEADERS` support authentication |
| 21 | `OUTPUT_DIR = "content/gaapo_outputs"` while the docs said `/content/gaapo_outputs` | Output directory is resolved against the kernel's start directory (or `GAAPO_OUTPUT_DIR`), printed as an absolute path, and defaults to a **new** directory so the original artifacts are preserved |
| 22 | No run provenance | `run_manifest.json` (versions, models, seeds, config, split hashes, request/token counts, parse breakdown) and a dated `report.md` |
| 23 | `message.content` only | Reasoning-channel and `finish_reason="length"` handling, empty/truncated counters, single-flight cache, transient-only retries |
| 24 | Dead `successive_halving`/`bandit` (no cell ran a standalone main experiment) | A standalone main-run cell, an executable smoke test, and an optional selection-method comparison |

**Unchanged on purpose:** the five strategies and their 40/20/20/10/10 weights, the eight mutation
types, the 50/50/200 split with `RANDOM_SEED=42`, the seed prompt text, the strict set-equality
metric, the OPRO trajectory without dropout, and the held-out test set's role as a final report
only.

## 16. Reproducing this

**Requirements.** macOS/Linux with [Ollama](https://ollama.com) running locally, Python ≥ 3.10, and

```bash
ollama pull qwen2.5:7b      # optimizer
ollama pull qwen2.5:3b      # target
```

or set `GAAPO_OPTIMIZER_MODEL` / `GAAPO_TARGET_MODEL` (any Ollama model; non-thinking instruct
models are the least surprising choice for the target role). GPU is not required — a 16 GB M1 runs
the defaults, slowly.

**Order.** Run the cells top to bottom from a fresh kernel. Nothing depends on state left over from
a previous execution: `run_gaapo` returns its own results, and the clients are re-created in
section 5.

**Stages** (all switches are in section 2):

| Stage | Switches | Section | Cost (defaults, measured scale) |
|---|---|---|---|
| Smoke test | `RUN_SMOKE_TEST` | 10 | ~4 min (measured) |
| Main run | `RUN_MAIN_EXPERIMENT`, `MAIN_*` | 11 | ~1.5–2 h |
| Ablation | `RUN_ABLATION`, `ABLATION_*` | 12 | ~7–9 h (4 arms × 3 repeats) |
| Adaptive study | `RUN_ADAPTIVE_STUDY`, `ADAPTIVE_*`, `AOS_*` | 17–21 | ~5–7 h (4 arms × 3 repeats; the `fixed` arm reuses the ablation's `full_hybrid` runs) |
| Selection comparison | `RUN_SELECTION_COMPARISON` | 13 | 3 main runs |
| Single-strategy baselines | `INCLUDE_SINGLE_STRATEGY` | 12 | +5 runs per repeat |

One full pass with the defaults is roughly **9–13 h** on an M1 with a 7B optimizer, so run it
overnight. `ABLATION_CONFIGS_TO_RUN` is trimmed to the four informative arms; list all six there if
you have the time. Dropping `ABLATION_N_REPEATS` to 2 is the cheapest way to shorten the pass, at
the cost of a weaker stability estimate.

**Cost model.** With `ALL` evaluation, target calls per run ≈ `50 + generations × population × 50`,
plus ~1 optimizer call per candidate and ~50 target calls per newly seen parent for APO's train
errors. Optimizer calls dominate: 20–30 s each for a 7B model on an M1 CPU against ~1–2 s for a 3B
target call. Generation is therefore dispatched concurrently (4 slots, `MAX_CONCURRENT_REQUESTS`)
with a deterministic per-slot RNG, so parallelism changes wall time but not the experiment. Set
`OLLAMA_NUM_PARALLEL=4` on the Ollama host to let the server actually run those requests at once;
with the default of 1 they queue and the sweep takes about twice as long. Caches collapse repeated
identical requests — across arms as well as within a run — so the real request count is well below
the nominal product (a measured smoke run issued 74 requests for 10 × 2 candidate evaluations).

**Determinism.** Data split, strategy order, parent sampling, few-shot example choice and the
ablation seeds all derive from `RANDOM_SEED`. The *models* are not bit-reproducible: local inference
varies with batch scheduling and quantisation, and the paper itself notes that evaluations had to be
repeated because of LLM response disparity. Same seed ⇒ same experiment structure, not identical
scores.

**Where results go.** `GAAPO_OUTPUT_DIR` (default `<notebook dir>/content/gaapo_outputs_v2`):
`candidate_log.csv`, `best_prompt.json`, `report.md`, `case_study_report.md`,
`run_manifest.json`, `figures/`, `ablation/`, `adaptive/`, `selection/`, `smoke/`. The original
`content/gaapo_outputs/` is never written to. `adaptive/` holds the per-run table, the per-example
test predictions, the weight trajectory, and the McNemar/bootstrap tables.

**Known limitations.** The ETHOS multilabel split used here has 433 rows (the paper says 443) and
the 50/50/200 partition is shuffled rather than stratified; the test set is drawn from the same
HuggingFace `train` split rather than a dataset-level held-out split; small local models are much
weaker optimizers than the paper's DeepSeek-R1-distill-70B, so absolute scores are far below the
published ones; macro-F1 and the selection-method comparison are diagnostics, not part of the
paper's protocol.

## 17. Adaptive operator selection

The paper fixes the five strategy weights at **40/20/20/10/10** and never justifies them.
Section 2.3.3.2 only says how weights become candidate counts — not where the numbers came from.
This section replaces the constant vector with a controller that **learns the allocation online
from the same validation signal**, and compares it against three controls at equal LLM budget.

This is *adaptive operator selection* (AOS), a standard topic in evolutionary computation:
credit assignment over variation operators plus an operator-selection rule (probability matching,
adaptive pursuit, dynamic multi-armed bandits). GAAPO is a natural fit because the five strategies
are exactly five operators competing for a fixed number of offspring per generation.

### Credit

After each generation, every child that was actually generated by strategy $s$ contributes

$$r = \text{score}(c) - \max_{p \in \text{parents}(c)} \text{score}(p)$$

the improvement over the best parent it was derived from. Rejected candidates contribute **0**
(they are a real failure of the operator, but they are reported separately so format failures are
not confused with prompt-quality failures). Credit is averaged per strategy and smoothed with an
exponential moving average, because the search is non-stationary: what works in generation 2 is not
what works in generation 6.

$$Q_s \leftarrow (1-\alpha)Q_s + \alpha\, r_s, \qquad \alpha = 0.30$$

### Selection

$$p_s = (1-\gamma)\,\frac{e^{Q_s/\tau}}{\sum_j e^{Q_j/\tau}} + \frac{\gamma}{K},
\qquad \tau = 0.50,\ \gamma = 0.10$$

initialised with $Q_s = \tau \log w_s^{\text{paper}}$, so the first generation reproduces the
paper's prior and every later change is attributable to observed credit. $Q$ starts on the prior's
log scale and converges to the credit scale ($|r| \le 1$) within a few updates — the *softmax
over Q*, not the absolute level, is what drives allocation. The quotas reuse the
paper's own rule (weight × population, largest remainder), with a floor of one slot per strategy so
no operator can be driven extinct by an early unlucky generation.

### Controls

| Arm | Allocation | Question it answers |
|---|---|---|
| `fixed` | 40/20/20/10/10, constant | the paper's default |
| `uniform` | 20 each, constant | is the paper's *prior* doing any work? |
| `random` | Dirichlet(1) redrawn each generation | is any gain just noise injection / extra exploration? |
| `adaptive` | credit-driven (above) | does *learning the allocation* help? |

`random` is the important control: without it, an adaptive win could simply mean "varying the
allocation helps", not "credit assignment helps".

### Hypotheses (fix these before looking at the results)

- **H1** adaptive ≥ fixed on best validation, and reaches its best in fewer generations.
- **H2** adaptive > random, i.e. the gain comes from credit assignment, not from variation.
- **H3** adaptation concentrates the population; we measure the normalised entropy of the weights
  and report any collapse.
- **H4** the adaptive-vs-fixed difference on the held-out test set is significant under a paired
  per-example test (McNemar) — this may well fail, see the note on power in section 20.

The controller reads **validation only**. The test set is touched once per run, for the final
comparison.

In [ ]:
import math


class BaseController:
    name = "base"

    def __init__(self, prior, rng):
        self.prior = dict(prior)
        self.rng = rng
        self.history = []

    def next_weights(self):
        raise NotImplementedError

    def observe(self, children):
        # children: list of {strategy, score, parent_scores, rejected}
        pass


class FixedController(BaseController):
    name = "fixed"

    def next_weights(self):
        weights = dict(self.prior)
        self.history.append({"generation": len(self.history) + 1,
                             "weights": weights, "credits": {}, "rejected": {}})
        return weights


class UniformController(BaseController):
    name = "uniform"

    def next_weights(self):
        k = len(self.prior)
        weights = {s: 1.0 / k for s in self.prior}
        self.history.append({"generation": len(self.history) + 1,
                             "weights": weights, "credits": {}, "rejected": {}})
        return weights


class RandomController(BaseController):
    name = "random"

    def next_weights(self):
        # Dirichlet(1) = normalised independent Exp(1) draws, from the controller's own RNG.
        draws = {s: self.rng.expovariate(1.0) for s in self.prior}
        total = sum(draws.values())
        weights = {s: v / total for s, v in draws.items()}
        self.history.append({"generation": len(self.history) + 1,
                             "weights": weights, "credits": {}, "rejected": {}})
        return weights


class AdaptiveController(BaseController):
    name = "adaptive"

    def __init__(self, prior, rng, alpha=AOS_ALPHA, tau=AOS_TAU, gamma=AOS_GAMMA):
        super().__init__(prior, rng)
        self.alpha, self.tau, self.gamma = alpha, tau, gamma
        # Q = tau * log(prior) makes the initial softmax reproduce the paper's weights.
        self.Q = {s: tau * math.log(max(w, 1e-9)) for s, w in self.prior.items()}

    def next_weights(self):
        k = len(self.Q)
        qmax = max(self.Q.values())
        exps = {s: math.exp((q - qmax) / self.tau) for s, q in self.Q.items()}
        z = sum(exps.values())
        weights = {s: (1.0 - self.gamma) * exps[s] / z + self.gamma / k for s in self.Q}
        self.history.append({"generation": len(self.history) + 1,
                             "weights": dict(weights), "credits": {}, "rejected": {}})
        return weights

    def observe(self, children):
        sums, counts, rejected = {}, {}, {}
        for child in children:
            s = child["strategy"]
            if s not in self.Q:
                continue
            if child["rejected"]:
                rejected[s] = rejected.get(s, 0) + 1
                continue
            parents = child.get("parent_scores") or []
            if not parents:
                continue
            r = child["score"] - max(parents)      # improvement over the best parent
            sums[s] = sums.get(s, 0.0) + r
            counts[s] = counts.get(s, 0) + 1
        credits = {s: sums[s] / counts[s] for s in sums}
        for s, r in credits.items():
            self.Q[s] = (1.0 - self.alpha) * self.Q[s] + self.alpha * r
        if self.history:
            self.history[-1]["credits"] = credits
            self.history[-1]["rejected"] = dict(rejected)


def make_controller(kind, prior, rng, **kwargs):
    if kind == "fixed":
        return FixedController(prior, rng)
    if kind == "uniform":
        return UniformController(prior, rng)
    if kind == "random":
        return RandomController(prior, rng)
    if kind == "adaptive":
        return AdaptiveController(prior, rng, **kwargs)
    raise ValueError(f"Unknown controller: {kind}")


def normalized_entropy(weights):
    values = [max(0.0, v) for v in weights.values()]
    total = sum(values)
    k = len(values)
    if k <= 1 or total <= 0:
        return 0.0
    probs = [v / total for v in values]
    return -sum(p * math.log(p) for p in probs if p > 0) / math.log(k)


# ---- controller self-checks
_rng = random.Random(0)
_ctl = AdaptiveController(STRATEGY_WEIGHTS, _rng)
assert Counter(allocate_candidates(_ctl.next_weights(), 10, floor=1)) == \
       Counter({"random_mutation": 4, "apo": 2, "opro": 2, "fewshot": 1, "crossover": 1}), \
       "the adaptive prior must reproduce the paper's quotas"
_before = dict(_ctl.Q)
_ctl.observe([{"strategy": "apo", "score": 0.5, "parent_scores": [0.2],
               "rejected": False}] * 5)
assert _ctl.Q["apo"] > _before["apo"], "positive credit must raise Q"
_opro_before = _ctl.Q["opro"]
_ctl.observe([{"strategy": "opro", "score": 0.0, "parent_scores": [0.4],
               "rejected": False}] * 5)
assert abs(_ctl.Q["opro"] - (-0.4)) < abs(_opro_before - (-0.4)), \
    "Q must move toward the observed credit"
assert abs(normalized_entropy({k: 0.2 for k in "abcde"}) - 1.0) < 1e-9
assert normalized_entropy({"a": 1.0, "b": 0.0}) == 0.0
print("controller self-checks passed")

In [ ]:
import numpy as np
import time

adaptive_df, adaptive_runs = None, {}

if RUN_ADAPTIVE_STUDY:
    ADAPTIVE_ARMS = ["fixed", "uniform", "random", "adaptive"]
    _dir = OUTPUT_DIR / "adaptive"
    _dir.mkdir(parents=True, exist_ok=True)
    runs_path = _dir / "adaptive_runs.csv"
    preds_path = _dir / "test_predictions.csv"
    traj_path = _dir / "weight_trajectory.csv"
    curve_path = _dir / "best_so_far.csv"

    rows, pred_rows, traj_rows, curve_rows = [], [], [], []
    completed = set()
    if RESUME_ADAPTIVE and runs_path.exists():
        rows = pd.read_csv(runs_path).to_dict("records")
        completed = {(str(r["arm"]), int(r["repeat"])) for r in rows}
        if preds_path.exists():
            pred_rows = pd.read_csv(preds_path).to_dict("records")
        if traj_path.exists():
            traj_rows = pd.read_csv(traj_path).to_dict("records")
        if curve_path.exists():
            curve_rows = pd.read_csv(curve_path).to_dict("records")
        print(f"resume: {len(completed)} arm-runs already on disk")

    def flush():
        # Written after every arm, so a crash or shutdown costs at most one run.
        pd.DataFrame(rows).to_csv(runs_path, index=False)
        pd.DataFrame(pred_rows).to_csv(preds_path, index=False)
        pd.DataFrame(traj_rows).to_csv(traj_path, index=False)
        pd.DataFrame(curve_rows).to_csv(curve_path, index=False)

    for repeat in range(ADAPTIVE_N_REPEATS):
        seed = RANDOM_SEED + 1000 * repeat          # paired across arms
        for arm in ADAPTIVE_ARMS:
            if (arm, repeat) in completed:
                print(f"skip arm {arm} repeat {repeat + 1}: already on disk")
                continue

            run, reused = None, False

            # The audit's full_hybrid runs are exactly the "fixed" arm at the same
            # scale and seed, so reuse them when they are in memory.
            if arm == "fixed" and ("full_hybrid", repeat) in ablation_runs:
                cand = ablation_runs[("full_hybrid", repeat)]
                m = cand["meta"]
                if (m["population_size"] == ADAPTIVE_POPULATION_SIZE
                        and m["generations"] == ADAPTIVE_GENERATIONS
                        and m["seed"] == seed
                        and m["eval_mode"] == ADAPTIVE_EVAL_MODE):
                    run, reused = cand, True
                    print(f"\n=== adaptive arm fixed  repeat {repeat + 1}: "
                          f"reused full_hybrid run ===")

            if run is None:
                print(f"\n=== adaptive arm {arm}  repeat {repeat + 1}/"
                      f"{ADAPTIVE_N_REPEATS} (seed={seed}) ===")
                controller = make_controller(
                    arm, STRATEGY_WEIGHTS, random.Random(seed ^ 0xA05),
                    alpha=AOS_ALPHA, tau=AOS_TAU, gamma=AOS_GAMMA)
                run = run_gaapo(
                    train_df, val_df,
                    population_size=ADAPTIVE_POPULATION_SIZE,
                    generations=ADAPTIVE_GENERATIONS,
                    eval_mode=ADAPTIVE_EVAL_MODE,
                    optimizer_client=optimizer_client, target_client=target_client,
                    seed=seed, weight_controller=controller, verbose=True,
                )

            best = run["hall_of_fame"]["best"]
            test_res = evaluate_on_examples(best["prompt"], test_df, target_client)
            adaptive_runs[(arm, repeat)] = run

            hist = run.get("weight_history") or []
            if not hist and reused:
                hist = [{"generation": g, "weights": dict(STRATEGY_WEIGHTS),
                         "credits": {}, "rejected": {}}
                        for g in range(1, ADAPTIVE_GENERATIONS + 1)]
            entropies = [normalized_entropy(h["weights"]) for h in hist]

            rows.append({
                "arm": arm, "repeat": repeat, "seed": seed, "reused": reused,
                "best_val": best["score"],
                "best_generation": best["generation"],
                "best_strategy": best["strategy"],
                "test_score": test_res.score,
                "test_macro_f1": test_res.macro_f1,
                "entropy_mean": float(np.mean(entropies)) if entropies else float("nan"),
                "entropy_final": float(entropies[-1]) if entropies else float("nan"),
                "requests": run["meta"]["requests_total"],
                "elapsed_s": run["meta"]["elapsed_s"],
                "final_weights": json.dumps(hist[-1]["weights"] if hist else {}),
            })
            for i, (p, g) in enumerate(zip(test_res.predictions, test_res.golds)):
                pred_rows.append({"arm": arm, "repeat": repeat, "example": i,
                                  "pred": json.dumps(p), "gold": json.dumps(g)})
            for h in hist:
                for strategy in STRATEGY_ORDER:
                    traj_rows.append({
                        "arm": arm, "repeat": repeat, "generation": h["generation"],
                        "strategy": strategy,
                        "weight": h["weights"].get(strategy, 0.0),
                        "credit": h.get("credits", {}).get(strategy, None),
                        "rejected": h.get("rejected", {}).get(strategy, 0),
                    })

            log = pd.DataFrame(run["candidate_log"])
            log.to_csv(_dir / f"log_{arm}_r{repeat}.csv", index=False)
            best_so_far = log.groupby("generation")["validation_score"].max().cummax()
            for g, v in best_so_far.items():
                curve_rows.append({"arm": arm, "repeat": repeat,
                                   "generation": int(g), "best_so_far": float(v)})

            flush()
            print(f"  -> best_val={best['score']:.3f} (gen {best['generation']}, "
                  f"{best['strategy']})  test={test_res.score:.3f}  "
                  f"entropy={rows[-1]['entropy_mean']:.3f}")

    adaptive_df = pd.DataFrame(rows)
    flush()
    print("\nadaptive study finished")
else:
    print("RUN_ADAPTIVE_STUDY = False; skipped")

In [ ]:
import matplotlib.pyplot as plt
from math import comb

adaptive_summary, adaptive_stats = None, None


def mcnemar_exact(a_correct, b_correct):
    # 2x2 McNemar on paired per-example correctness; exact two-sided binomial.
    b = sum(1 for x, y in zip(a_correct, b_correct) if x and not y)
    c = sum(1 for x, y in zip(a_correct, b_correct) if (not x) and y)
    n = b + c
    if n == 0:
        return {"b": 0, "c": 0, "p": 1.0}
    k = min(b, c)
    p = 2.0 * sum(comb(n, i) for i in range(k + 1)) / (2 ** n)
    return {"b": b, "c": c, "p": min(1.0, p)}


def holm_adjust(pvalues):
    m = len(pvalues)
    order = sorted(range(m), key=lambda i: pvalues[i])
    adjusted = [1.0] * m
    running = 0.0
    for rank, i in enumerate(order):
        value = min(1.0, (m - rank) * pvalues[i])
        running = max(running, value)
        adjusted[i] = running
    return adjusted


def paired_bootstrap_ci(a_correct, b_correct, n_boot=10000, seed=0):
    # Bootstrap over examples: CI for the paired accuracy difference A - B.
    a = np.asarray(a_correct, dtype=float)
    b = np.asarray(b_correct, dtype=float)
    d = a - b
    if len(d) == 0:
        return (float("nan"), float("nan"))
    rng_boot = np.random.default_rng(seed)
    idx = rng_boot.integers(0, len(d), size=(n_boot, len(d)))
    means = d[idx].mean(axis=1)
    return float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5))


if adaptive_df is not None:
    _dir = OUTPUT_DIR / "adaptive"
    summary = (adaptive_df.groupby("arm")
               .agg(best_val_mean=("best_val", "mean"),
                    best_val_std=("best_val", "std"),
                    test_mean=("test_score", "mean"),
                    test_std=("test_score", "std"),
                    entropy_mean=("entropy_mean", "mean"),
                    best_gen_mean=("best_generation", "mean"),
                    n=("repeat", "count"))
               .reset_index())
    fixed_row = summary.loc[summary["arm"] == "fixed"].iloc[0]
    summary["delta_val_vs_fixed"] = summary["best_val_mean"] - fixed_row["best_val_mean"]
    summary["delta_test_vs_fixed"] = summary["test_mean"] - fixed_row["test_mean"]
    adaptive_summary = summary.sort_values("test_mean", ascending=False)

    # ---- paired per-example statistics on the held-out test set
    preds = pd.read_csv(_dir / "test_predictions.csv")
    preds["correct"] = [set(json.loads(p)) == set(json.loads(g))
                        for p, g in zip(preds["pred"], preds["gold"])]
    stat_rows = []
    for repeat in sorted(preds["repeat"].unique()):
        a = preds[(preds["arm"] == "adaptive") & (preds["repeat"] == repeat)]
        a = a.sort_values("example")
        for control in ("fixed", "uniform", "random"):
            b = preds[(preds["arm"] == control) & (preds["repeat"] == repeat)]
            b = b.sort_values("example")
            ca, cb = a["correct"].tolist(), b["correct"].tolist()
            mc = mcnemar_exact(ca, cb)
            lo, hi = paired_bootstrap_ci(ca, cb)
            stat_rows.append({
                "repeat": repeat, "contrast": f"adaptive_vs_{control}",
                "acc_adaptive": float(np.mean(ca)), "acc_control": float(np.mean(cb)),
                "diff": float(np.mean(ca) - np.mean(cb)),
                "ci_low": lo, "ci_high": hi,
                "b_adaptive_only": mc["b"], "c_control_only": mc["c"],
                "p_raw": mc["p"],
            })
    stats = pd.DataFrame(stat_rows)
    stats["p_holm"] = holm_adjust(stats["p_raw"].tolist())
    adaptive_stats = stats
    stats.to_csv(_dir / "stats_mcnemar.csv", index=False)
    stats[["repeat", "contrast", "diff", "ci_low", "ci_high"]] \
        .rename(columns={"diff": "paired_diff", "ci_low": "boot_ci_low",
                         "ci_high": "boot_ci_high"}) \
        .to_csv(_dir / "stats_bootstrap.csv", index=False)
    adaptive_summary.to_csv(_dir / "adaptive_summary.csv", index=False)

    print(adaptive_summary.round(3).to_string(index=False))
    print()
    print(stats.round(4).to_string(index=False))

    # ---- figures
    fig_dir = OUTPUT_DIR / "figures"
    fig_dir.mkdir(parents=True, exist_ok=True)
    traj = pd.read_csv(_dir / "weight_trajectory.csv")

    fig, ax = plt.subplots(figsize=(9, 5))
    adaptive_traj = traj[traj["arm"] == "adaptive"]
    for strategy in STRATEGY_ORDER:
        block = adaptive_traj[adaptive_traj["strategy"] == strategy]
        pivot = block.pivot_table(index="generation", columns="repeat", values="weight")
        if pivot.empty:
            continue
        ax.plot(pivot.index, pivot.mean(axis=1), marker="o", label=strategy)
        ax.fill_between(pivot.index, pivot.min(axis=1), pivot.max(axis=1), alpha=0.15)
    ax.set_xlabel("Generation")
    ax.set_ylabel("Strategy weight")
    ax.set_title("Adaptive operator selection: weights over generations "
                 "(mean, band = min/max over repeats)")
    ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(fig_dir / "adaptive_weights.png", dpi=150)
    plt.show()

    fig, ax = plt.subplots(figsize=(7, 4.5))
    for arm in ADAPTIVE_ARMS:
        block = traj[traj["arm"] == arm]
        pivot = block.pivot_table(index="generation", columns="repeat", values="weight")
        if pivot.empty:
            continue
        ent = [normalized_entropy(pivot.loc[g].to_dict()) for g in pivot.index]
        ax.plot(pivot.index, ent, marker="o", label=arm)
    ax.set_xlabel("Generation")
    ax.set_ylabel("Normalised weight entropy")
    ax.set_ylim(0, 1.05)
    ax.set_title("Allocation diversity (1 = uniform, 0 = one strategy)")
    ax.legend()
    fig.tight_layout()
    fig.savefig(fig_dir / "adaptive_entropy.png", dpi=150)
    plt.show()

    curves = pd.read_csv(_dir / "best_so_far.csv")
    fig, ax = plt.subplots(figsize=(8, 4.8))
    for arm in ADAPTIVE_ARMS:
        block = curves[curves["arm"] == arm]
        if block.empty:
            continue
        pivot = block.pivot_table(index="generation", columns="repeat",
                                  values="best_so_far")
        ax.plot(pivot.index, pivot.mean(axis=1), marker="o", label=arm)
        ax.fill_between(pivot.index, pivot.min(axis=1), pivot.max(axis=1), alpha=0.12)
    ax.set_xlabel("Generation")
    ax.set_ylabel("Best validation score so far")
    ax.set_title("Sample efficiency by allocation policy")
    ax.legend()
    fig.tight_layout()
    fig.savefig(fig_dir / "adaptive_best_so_far.png", dpi=150)
    plt.show()

    # ---- plain-language verdict
    print()
    print("--- adaptive study, plain language ---")
    print(f"repeats: {ADAPTIVE_N_REPEATS}; scale: "
          f"{ADAPTIVE_POPULATION_SIZE} x {ADAPTIVE_GENERATIONS}")
    print(adaptive_summary[["arm", "best_val_mean", "best_val_std",
                            "test_mean", "test_std", "entropy_mean"]]
          .round(3).to_string(index=False))
    sig = stats[stats["p_holm"] < 0.05]
    if len(sig):
        print("Holm-corrected significant test differences: "
              + ", ".join(f"{r.contrast} (repeat {r.repeat}, "
                          f"{r.diff:+.3f})" for r in sig.itertuples()))
    else:
        print("No Holm-corrected significant test difference. With this many "
              "examples the study cannot resolve small effects; report it as "
              "inconclusive rather than as evidence for or against adaptation.")
    print("If the within-arm spread across repeats is comparable to the "
          "between-arm differences, treat the ranking as unresolved.")
else:
    print("No adaptive results (RUN_ADAPTIVE_STUDY = False)")

In [ ]:
from datetime import datetime, timezone

if adaptive_df is None:
    print("No adaptive results: case_study_report.md not written")
else:
    def _md_table(frame):
        try:
            return frame.to_markdown(index=False)
        except Exception:
            return frame.to_string(index=False)

    lines = []
    lines.append("# GAAPO case study — reproducibility audit and adaptive operator selection\n")
    lines.append(f"_generated {datetime.now(timezone.utc).isoformat(timespec='seconds')}_\n")
    lines.append(f"- optimizer: `{OPTIMIZER_MODEL}`")
    lines.append(f"- target: `{TARGET_MODEL}`")
    lines.append(f"- seed: {RANDOM_SEED}")
    lines.append(f"- split: {SPLIT_MANIFEST['n_train']} train / "
                 f"{SPLIT_MANIFEST['n_val']} val / {SPLIT_MANIFEST['n_test']} test\n")

    lines.append("## 1. Reproducibility audit\n")
    if log_df is not None:
        lines.append(f"- seed prompt (full validation): **{baseline_val_score:.3f}**")
        lines.append(f"- best evolved prompt (full validation): **{best_val_score:.3f}** "
                     f"(generation {best['generation']}, `{best['strategy']}`)")
        lines.append(f"- held-out test: **{test_res.score:.3f}** "
                     f"(baseline {baseline_test.score:.3f})\n")
        lines.append("The ablation and its error bars are in section 12 and "
                     "`ablation/ablation_summary.csv`.\n")
    elif (OUTPUT_DIR / "candidate_log.csv").exists():
        stored = pd.read_csv(OUTPUT_DIR / "candidate_log.csv")
        stored_best = stored.loc[stored["validation_score"].idxmax()]
        lines.append(f"- stored main run: {len(stored)} logged candidates over "
                     f"{stored['generation'].nunique()} generations")
        lines.append(f"- best validation in that run: "
                     f"**{stored_best['validation_score']:.3f}** "
                     f"(generation {int(stored_best['generation'])}, "
                     f"`{stored_best['strategy']}`)\n")
        audit_md = OUTPUT_DIR / "report.md"
        if audit_md.exists():
            lines.append(audit_md.read_text())
            lines.append("")
    else:
        lines.append("Audit stages were skipped in this session; see `report.md` "
                     "and the stored artifacts.\n")

    lines.append("## 2. Adaptive operator selection\n")
    lines.append("Four allocation policies at equal LLM budget: fixed (paper weights), "
                 "uniform, random (Dirichlet redrawn each generation), and adaptive "
                 "(EWMA credit + softmax + exploration floor).\n")
    lines.append(_md_table(adaptive_summary.round(3)))
    lines.append("")
    lines.append("### Paired per-example comparison on the held-out test set\n")
    lines.append(_md_table(adaptive_stats.round(4)))
    lines.append("")
    lines.append("`b` = adaptive correct and control wrong; `c` = control correct and "
                 "adaptive wrong; `p_holm` is the Holm-adjusted exact McNemar p-value; "
                 "the CI is a paired bootstrap over the "
                 f"{SPLIT_MANIFEST['n_test']} test examples.\n")
    lines.append("### Figures\n")
    lines.append("- `figures/adaptive_weights.png` — learned weights over generations")
    lines.append("- `figures/adaptive_entropy.png` — allocation diversity per arm")
    lines.append("- `figures/adaptive_best_so_far.png` — sample efficiency\n")

    lines.append("## 3. Limitations\n")
    lines.append(f"- Small local models, {SPLIT_MANIFEST['n_val']}-example validation: the "
                 "credit signal is noisy and the study is underpowered for small effects.")
    lines.append("- Three repeats are a stability check, not a power source; the primary "
                 "inference is the paired per-example test.")
    lines.append("- The test set is used for reporting, not for selecting the arm. Do not "
                 "pick the best arm by test score and call it a finding.")
    lines.append("- Absolute scores are not comparable with the published paper (different "
                 "models, smaller budget).\n")

    path = OUTPUT_DIR / "case_study_report.md"
    path.write_text("\n".join(lines))
    print("wrote", path)